In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1: ENVIRONMENT SETUP + DEEP EXPLORATION
# ══════════════════════════════════════════════════════════════════════
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │                    WHAT THIS CELL DOES                             │
# │                                                                    │
# │  1. Clones the LightFC repo from GitHub                            │
# │  2. Installs dependencies (thop, easydict, timm, etc.)             │
# │  3. Finds and links the pretrained checkpoint                      │
# │  4. Patches compatibility issues (torch._six, weights_only, etc.)  │
# │  5. Explores EVERY component we'll use in later cells              │
# │  6. Runs a 10-frame tracking test to verify the pipeline           │
# └─────────────────────────────────────────────────────────────────────┘
#
# ── REPO STRUCTURE ───────────────────────────────────────────────────
#
#   /kaggle/working/LightFC/
#   ├── experiments/lightfc/
#   │   └── baseline_v1_release_backbone_tinyvit.yaml  ← CONFIG FILE
#   │       Defines ALL hyperparameters:
#   │         TEMPLATE: 128px, factor=2.0
#   │         SEARCH:   256px, factor=4.0
#   │         STRIDE:   16 (spatial downscale ratio)
#   │         HEAD:     inplanes=96, channel=256, feat_sz=16
#   │
#   ├── lib/
#   │   ├── models/
#   │   │   ├── tracker_model.py          ← MAIN MODEL CLASS (LightFC)
#   │   │   │   Contains: backbone + fusion + head
#   │   │   │   forward():      training path (both through backbone)
#   │   │   │   forward_backbone(): template feature extraction
#   │   │   │   forward_tracking(): search tracking (backbone→fusion→head)
#   │   │   │
#   │   │   └── lightfc/
#   │   │       ├── backbone/             ← TinyViT-5M (2.44M params)
#   │   │       │   Input:  (B,3,H,W) → Output: (B,160,H/16,W/16)
#   │   │       │   Template: (1,3,128,128) → (1,160,8,8)
#   │   │       │   Search:   (1,3,256,256) → (1,160,16,16)
#   │   │       │
#   │   │       ├── fusion/               ← pwcorr_se (0.03M params)
#   │   │       │   Pixel-wise correlation between z_feat and x_feat
#   │   │       │   "Where in the search does the template match?"
#   │   │       │   Input: z(1,160,8,8) + x(1,160,16,16)
#   │   │       │   Output: (1,96,16,16) fused correlation features
#   │   │       │
#   │   │       └── head/                 ← repn33_se_center (4.81M params)
#   │   │           THREE parallel branches:
#   │   │             score_map  (1,1,16,16)  → "Where is target center?"
#   │   │             size_map   (1,2,16,16)  → "How big is target?"
#   │   │             offset_map (1,2,16,16)  → "Sub-pixel offset?"
#   │   │             pred_boxes (1,4)        → decoded (cx,cy,w,h)
#   │   │           Uses RepVGG blocks:
#   │   │             TRAINING:  rbr_dense + rbr_3x3 (parallel paths)
#   │   │             INFERENCE: rbr_reparam (single fused conv)
#   │   │
#   │   ├── train/data/
#   │   │   └── processing_utils.py       ← CROP FUNCTIONS
#   │   │       sample_target():
#   │   │         crop_sz = ceil(sqrt(w*h) * factor)
#   │   │         Extracts square crop, pads if needed, resizes
#   │   │         Returns: (crop, resize_factor, attention_mask)
#   │   │       transform_image_to_crop():
#   │   │         Converts bbox from image coords → crop coords
#   │   │         Used to create training labels
#   │   │
#   │   ├── test/
#   │   │   ├── tracker/
#   │   │   │   ├── lightfc.py            ← NATIVE TRACKER CLASS
#   │   │   │   │   initialize(): crop template → backbone features
#   │   │   │   │   track(): crop search → full decode pipeline
#   │   │   │   │   compute_box(): cal_bbox → scale by resize_factor
#   │   │   │   │   map_box_back(): crop coords → image coords
#   │   │   │   │
#   │   │   │   └── data_utils.py         ← PREPROCESSOR
#   │   │   │       Preprocessor.process(img, mask):
#   │   │   │         img/255 → ImageNet normalize → CUDA tensor
#   │   │   │         Returns NestedTensor(.tensors, .mask)
#   │   │   │
#   │   │   └── utils/
#   │   │       └── hann.py               ← HANNING WINDOW
#   │   │           hann2d([16,16]): cosine window for score suppression
#   │   │           Center=1.0, edges→0.0
#   │   │           Penalizes detections far from previous position
#   │   │
#   │   └── utils/
#   │       └── box_ops.py                ← BOX UTILITIES
#   │           clip_box(): clamp bbox within image boundaries
#   │
#   └── output/checkpoints/               ← WHERE WE LINK CHECKPOINT
#
# ── DATA STRUCTURE ───────────────────────────────────────────────────
#
#   /kaggle/input/datasets/yussufothman/contest-tracking-data/
#   ├── metadata/
#   │   ├── contestant_manifest.json      ← ALL sequence info
#   │   │   {"train": {key: {video_path, annotation_path, n_frames}},
#   │   │    "public_lb": {key: {...}}}
#   │   └── sample_submission.csv         ← Template (74,293 rows)
#   │       Format: "dataset1/Car_video_0,0,0,0,0"
#   │       Parse:  rsplit('_', 1) → (seq_name, frame_idx)
#   │
#   ├── dataset1/ through dataset5/
#   │   └── <sequence_name>/
#   │       ├── <sequence_name>.mp4       ← Video file
#   │       └── annotation.txt            ← GT bboxes (x,y,w,h per line)
#   │
#   /kaggle/input/datasets/yussufothman/od-predictions/
#   ├── train/                            ← ODTrack predictions for 217 seqs
#   │   └── dataset1_Car_video_2.txt      ← "383.00,545.00,156.00,74.00"
#   └── test/                             ← ODTrack predictions for 89 seqs
#       └── dataset1_Car_video.txt
#
# ── CONFIG SYSTEM ────────────────────────────────────────────────────
#
#   The YAML file is loaded into a DotDict (nested dict with dot access):
#     cfg.MODEL.BACKBONE.TYPE → 'tiny_vit_5m_224'
#     cfg.DATA.SEARCH.SIZE → 256
#     cfg.MODEL.BACKBONE.STRIDE → 16
#
#   This cfg object is passed to:
#     LightFC(cfg) → decides backbone type, head params, etc.
#     Tracker → decides crop sizes, factors
#
# ── CHECKPOINT FORMAT ────────────────────────────────────────────────
#
#   lightfc_ep0400.pth.tar (94.6MB):
#     {'epoch': 400, 'net': state_dict, 'optimizer': ..., 'stats': ...}
#
#   state_dict has 393 keys:
#     backbone.* (236 keys, 2.44M params) — TinyViT conv+attention
#     fusion.*   (34 keys, 0.03M params)  — correlation layers
#     head.*     (123 keys, 4.81M params) — RepVGG prediction head
#
#   RepVGG keys in TRAINING mode:
#     head.conv1_ctr.rbr_dense.conv.weight  ← 3×3 main path
#     head.conv1_ctr.rbr_3x3.conv.weight    ← 3×3 shortcut path
#     head.conv1_ctr.rbr_dense.bn.*         ← BatchNorm stats
#
#   After switch_to_deploy():
#     head.conv1_ctr.rbr_reparam.weight     ← fused single conv
#     head.conv1_ctr.rbr_reparam.bias
#     (rbr_dense and rbr_3x3 are DELETED — cannot train after this!)
#
# ── FULL INFERENCE PIPELINE ──────────────────────────────────────────
#
#   Frame 0 (init):
#     annotation.txt line 1 → init_bbox [x,y,w,h]
#     sample_target(image, init_bbox, factor=2.0, sz=128) → template crop
#     Preprocessor → normalize → NestedTensor
#     forward_backbone(template) → z_feat (1,160,8,8)
#
#   Frame N (track):
#     sample_target(image, prev_state, factor=4.0, sz=256) → search crop
#     Preprocessor → normalize → NestedTensor
#     forward_tracking(z_feat, search) → {score_map, size_map, offset_map}
#     response = hann2d(16,16) × score_map    ← suppress off-center peaks
#     cal_bbox(response, size_map, offset_map) → [cx,cy,w,h] normalized
#     × 256 / resize_factor                   ← scale to real pixels
#     map_box_back(prev_state)                 ← crop coords → image coords
#     clip_box(H, W)                           ← clamp to image bounds
#     → new state [x,y,w,h]
#
# ══════════════════════════════════════════════════════════════
# CELL 1: Environment Setup + Full Exploration
# ══════════════════════════════════════════════════════════════
#
# This cell does TWO things:
#   PART 1: Environment setup (clone, patch, prepare)
#   PART 2: Deep exploration of EVERY component we'll use
#
# After this cell, we will print a FULL REPORT with:
#   - Exact sample_target source + crop formula
#   - Exact Preprocessor behavior
#   - Exact model output shapes/ranges
#   - Exact head.cal_bbox source + behavior
#   - Exact checkpoint format (deploy vs training RepVGG)
#   - Exact switch_to_deploy effect
#   - Exact submission CSV format + name mapping
#   - End-to-end 5-frame tracking test with IoU vs GT
#
# ══════════════════════════════════════════════════════════════

import os, sys, glob, re, shutil, types, json, csv, math, inspect, copy
import numpy as np
import cv2
import torch

# ─────────────────────────────────────────────────────────────
# PART 1: ENVIRONMENT SETUP
# ─────────────────────────────────────────────────────────────

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/LightFC"
DATA_ROOT = "/kaggle/input/datasets/yussufothman/contest-tracking-data"
LIGHTFC_INPUT = "/kaggle/input/datasets/yussufothman/lightfc-vit-model"
OD_PRED_DIR = "/kaggle/input/datasets/yussufothman/od-predictions"

os.chdir(WORK)

# ── 1a: Clone repo ───────────────────────────────────────────
if not os.path.exists(f"{REPO_DIR}/lib"):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    ret = os.system(f"git clone --depth 1 https://github.com/LiYunfengLYF/LightFC.git {REPO_DIR}")
    if ret != 0:
        os.system(f'wget -q -O {WORK}/lfc.zip '
                  '"https://github.com/LiYunfengLYF/LightFC/archive/refs/heads/main.zip"')
        import zipfile
        with zipfile.ZipFile(f"{WORK}/lfc.zip") as z:
            z.extractall(WORK)
        for d in glob.glob(f"{WORK}/LightFC-*"):
            shutil.move(d, REPO_DIR)
        os.remove(f"{WORK}/lfc.zip")

assert len(glob.glob(f"{REPO_DIR}/**/*.py", recursive=True)) > 5, "Clone failed"
print(f"✅ 1a: Repo ready: {REPO_DIR}")

# ── 1b: Install deps ─────────────────────────────────────────
os.chdir(REPO_DIR)
os.system("pip install -q thop easydict jpeg4py lmdb einops pyyaml timm==0.9.2 2>/dev/null")
sys.path.insert(0, REPO_DIR)
print("✅ 1b: Dependencies installed")

# ── 1c: Find checkpoint ──────────────────────────────────────
CKPT_PATH = None
for f in glob.glob(f"{LIGHTFC_INPUT}/**/*.pth.tar", recursive=True):
    CKPT_PATH = f; break
if not CKPT_PATH:
    for f in glob.glob(f"{LIGHTFC_INPUT}/**/*.pth", recursive=True):
        CKPT_PATH = f; break
assert CKPT_PATH, "❌ No checkpoint found!"
print(f"✅ 1c: Checkpoint: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)/1e6:.1f}MB)")

# ── 1d: Patch compatibility issues ───────────────────────────
# torch._six shim
if "torch._six" not in sys.modules:
    _six = types.ModuleType("torch._six")
    _six.string_classes = (str, bytes)
    sys.modules["torch._six"] = _six

# Patch .py files
patched = 0
for pyf in glob.glob(f"{REPO_DIR}/**/*.py", recursive=True):
    if "__pycache__" in pyf: continue
    txt = open(pyf).read()
    changed = False
    if "from torch._six import" in txt:
        txt = re.sub(r'from torch._six import .*', 'string_classes = (str, bytes)', txt)
        changed = True
    if "torch.load(" in txt and "weights_only" not in txt:
        txt = re.sub(r'torch\.load\(([^)]+)\)', r'torch.load(\1, weights_only=False)', txt)
        changed = True
    if changed:
        open(pyf, "w").write(txt)
        patched += 1

# Patch local.py paths
for lp in glob.glob(f"{REPO_DIR}/**/local.py", recursive=True):
    if "__pycache__" in lp: continue
    txt = open(lp).read()
    txt = re.sub(r"(self\.workspace_dir\s*=\s*).*", f"\\1'{REPO_DIR}'", txt)
    txt = re.sub(r"(self\.tensorboard_dir\s*=\s*).*", f"\\1'{REPO_DIR}/tensorboard'", txt)
    txt = re.sub(r"(settings\.prj_dir\s*=\s*).*", f"\\1'{REPO_DIR}'", txt)
    txt = re.sub(r"(settings\.save_dir\s*=\s*).*", f"\\1'{REPO_DIR}/output'", txt)
    txt = re.sub(r"/home/\w+/[\w/.-]+(?=/output|/experiments|/pretrained|')", REPO_DIR, txt)
    open(lp, "w").write(txt)

# Patch tracker file for safe attribute access
tf = f"{REPO_DIR}/lib/test/tracker/lightfc.py"
if os.path.exists(tf):
    txt = open(tf).read()
    for old, new in [
        ("self.debug = params.debug", "self.debug = getattr(params, 'debug', 0)"),
        ("self.use_visdom = params.debug", "self.use_visdom = getattr(params, 'debug', 0)"),
        ("self.use_visdom = params.use_visdom", "self.use_visdom = getattr(params, 'use_visdom', 0)"),
        ("params.visualization", "getattr(params, 'visualization', False)"),
        ("params.visdom_info", "getattr(params, 'visdom_info', {})"),
    ]:
        txt = txt.replace(old, new)
    open(tf, "w").write(txt)

# Global torch.load patch
import torch.serialization
_real_torch_load = torch.serialization.load
def _safe_load(*a, **kw):
    kw.setdefault("weights_only", False)
    return _real_torch_load(*a, **kw)
torch.load = _safe_load

# Clear caches
for d in glob.glob(f"{REPO_DIR}/**/__pycache__", recursive=True):
    shutil.rmtree(d)

print(f"✅ 1d: Patched {patched} files")

# ── 1e: Link checkpoint ──────────────────────────────────────
TRACKER_NAME = "lightfc"
CONFIG_NAME = "baseline_v1_release_backbone_tinyvit"
ckpt_dir = f"{REPO_DIR}/output/checkpoints/train/{TRACKER_NAME}/{CONFIG_NAME}"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_dst = f"{ckpt_dir}/lightfc_ep0400.pth.tar"
if os.path.exists(ckpt_dst): os.remove(ckpt_dst)
os.symlink(CKPT_PATH, ckpt_dst)
print(f"✅ 1e: Checkpoint linked")

print("\n" + "=" * 70)
print("PART 1 COMPLETE — ENVIRONMENT READY")
print("=" * 70)

# ─────────────────────────────────────────────────────────────
# PART 2: DEEP EXPLORATION
# ─────────────────────────────────────────────────────────────

# ══════════════════════════════════════════════════════════════
# EXPLORE A: sample_target source code + crop formula
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE A: sample_target")
print("=" * 70)

from lib.train.data.processing_utils import sample_target as repo_sample_target

src_file = inspect.getfile(repo_sample_target)
print(f"Module: {repo_sample_target.__module__}")
print(f"File:   {src_file}")
print(f"Signature: {inspect.signature(repo_sample_target)}")

with open(src_file) as f:
    full_src = f.read()
idx = full_src.find("def sample_target")
if idx >= 0:
    end = full_src.find("\ndef ", idx + 10)
    if end == -1: end = idx + 3000
    print("\n--- SOURCE CODE ---")
    print(full_src[idx:end])

# Also find transform_image_to_crop and any jittered_center_crop
for fname in ["def transform_image_to_crop", "def jittered_center_crop",
              "def sample_target_adaptive"]:
    idx2 = full_src.find(fname)
    if idx2 >= 0:
        end2 = full_src.find("\ndef ", idx2 + 10)
        if end2 == -1: end2 = idx2 + 2000
        print(f"\n--- {fname} ---")
        print(full_src[idx2:end2])

# ══════════════════════════════════════════════════════════════
# EXPLORE B: Preprocessor source
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE B: Preprocessor")
print("=" * 70)

from lib.test.tracker.data_utils import Preprocessor

pp_file = inspect.getfile(Preprocessor)
print(f"File: {pp_file}")
with open(pp_file) as f:
    print(f.read()[:3000])

# ══════════════════════════════════════════════════════════════
# EXPLORE C: Native tracker source (complete)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE C: Native tracker (lib/test/tracker/lightfc.py)")
print("=" * 70)

tracker_file = f"{REPO_DIR}/lib/test/tracker/lightfc.py"
with open(tracker_file) as f:
    print(f.read())

# ══════════════════════════════════════════════════════════════
# EXPLORE D: tracker_model.py (forward pass logic)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE D: tracker_model.py")
print("=" * 70)

model_file = f"{REPO_DIR}/lib/models/tracker_model.py"
with open(model_file) as f:
    print(f.read())

# ══════════════════════════════════════════════════════════════
# EXPLORE E: Head source (cal_bbox, forward, etc.)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE E: Head module source")
print("=" * 70)

for pyf in sorted(glob.glob(f"{REPO_DIR}/lib/models/**/head*.py", recursive=True)):
    if "__pycache__" in pyf: continue
    print(f"\n--- {pyf} ---")
    with open(pyf) as f:
        content = f.read()
    # Print full file if small, or key methods if large
    if len(content) < 5000:
        print(content)
    else:
        for kw in ['class ', 'def forward', 'def cal_bbox', 'def cal',
                    'def inference', 'def get_pred']:
            idx = content.find(kw)
            while idx >= 0:
                end = content.find("\n    def ", idx + 10)
                if end == -1: end = content.find("\nclass ", idx + 10)
                if end == -1: end = idx + 2000
                print(content[max(0,idx-50):end])
                print("...\n")
                idx = content.find(kw, end)

# ══════════════════════════════════════════════════════════════
# EXPLORE F: hann2d + clip_box + box_ops
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE F: hann2d + clip_box")
print("=" * 70)

from lib.test.utils.hann import hann2d
print(f"hann2d signature: {inspect.signature(hann2d)}")
hann_file = inspect.getfile(hann2d)
with open(hann_file) as f:
    print(f.read()[:2000])

from lib.utils.box_ops import clip_box
print(f"\nclip_box signature: {inspect.signature(clip_box)}")
box_file = inspect.getfile(clip_box)
with open(box_file) as f:
    box_src = f.read()
# Print clip_box function
idx = box_src.find("def clip_box")
if idx >= 0:
    end = box_src.find("\ndef ", idx + 10)
    if end == -1: end = idx + 500
    print(box_src[idx:end])

# ══════════════════════════════════════════════════════════════
# EXPLORE G: Checkpoint format + switch_to_deploy
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE G: Checkpoint format")
print("=" * 70)

ckpt_data = torch.load(CKPT_PATH, map_location='cpu')
print(f"Top-level keys: {list(ckpt_data.keys())}")
state = ckpt_data.get('net', ckpt_data)
print(f"State dict keys count: {len(state)}")

# RepVGG analysis
has_reparam = any('rbr_reparam' in k for k in state)
has_dense = any('rbr_dense' in k for k in state)
has_3x3 = any('rbr_3x3' in k for k in state)
has_1x1 = any('rbr_1x1' in k for k in state)
print(f"RepVGG: reparam={has_reparam}, dense={has_dense}, 3x3={has_3x3}, 1x1={has_1x1}")

# Print ALL keys grouped by module
for prefix in ['backbone', 'fusion', 'head']:
    keys = sorted([k for k in state if k.startswith(prefix)])
    print(f"\n{prefix} keys ({len(keys)}):")
    for k in keys[:20]:
        print(f"  {k}: {state[k].shape}")
    if len(keys) > 20:
        print(f"  ... ({len(keys)-20} more)")

# Find switch_to_deploy source
print(f"\nswitch_to_deploy source:")
for pyf in glob.glob(f"{REPO_DIR}/lib/**/*.py", recursive=True):
    if "__pycache__" in pyf: continue
    with open(pyf) as f:
        content = f.read()
    if "def switch_to_deploy" in content:
        idx = content.find("def switch_to_deploy")
        end = content.find("\n    def ", idx + 10)
        if end == -1:
            end = content.find("\nclass ", idx + 10)
        if end == -1:
            end = idx + 1500
        print(f"\n--- {pyf} ---")
        print(content[idx:end])

del ckpt_data, state

# ══════════════════════════════════════════════════════════════
# EXPLORE H: Submission CSV format
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE H: Submission CSV")
print("=" * 70)

SAMPLE_SUB_PATH = f"{DATA_ROOT}/metadata/sample_submission.csv"
MANIFEST_PATH = f"{DATA_ROOT}/metadata/contestant_manifest.json"

with open(SAMPLE_SUB_PATH) as f:
    reader = csv.reader(f)
    header = next(reader)
    all_rows = [row for row in reader]

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

test_manifest = manifest["public_lb"]
train_manifest = manifest["train"]

sample_ids = [r[0] for r in all_rows]

print(f"Header: {header}")
print(f"Total rows: {len(all_rows)}")
print(f"Column count: {len(all_rows[0])}")
print(f"\nFirst 10 rows:")
for r in all_rows[:10]:
    print(f"  {r}")
print(f"\nLast 5 rows:")
for r in all_rows[-5:]:
    print(f"  {r}")

# Try EVERY possible separator to parse row IDs
print(f"\nSeparator detection:")
for sep_name, sep in [("underscore _", "_"), ("dash -", "-"), 
                       ("colon :", ":"), ("comma ,", ","), ("dot .", ".")]:
    parts = sample_ids[0].rsplit(sep, 1)
    if len(parts) == 2 and parts[1].isdigit():
        # Count how many parse correctly
        ok = sum(1 for rid in sample_ids if len(rid.rsplit(sep,1))==2 and rid.rsplit(sep,1)[1].isdigit())
        print(f"  '{sep_name}': {ok}/{len(sample_ids)} parse OK")
        if ok == len(sample_ids):
            print(f"    ✅ WINNER! All rows parse with '{sep}'")
            all_sub_seqs = set(rid.rsplit(sep, 1)[0] for rid in sample_ids)
            print(f"    Unique seqs: {len(all_sub_seqs)}")
            print(f"    Samples: {sorted(all_sub_seqs)[:5]}")
    else:
        print(f"  '{sep_name}': first row doesn't parse")

# Map manifest keys to submission seq names
print(f"\nManifest→Submission mapping:")
print(f"  Manifest test keys (first 5): {list(test_manifest.keys())[:5]}")

# Determine which separator worked
SEPARATOR = None
for sep in ['_', '-', ':', ',', '.']:
    ok = sum(1 for rid in sample_ids if len(rid.rsplit(sep,1))==2 and rid.rsplit(sep,1)[1].isdigit())
    if ok == len(sample_ids):
        SEPARATOR = sep
        break

if SEPARATOR:
    all_sub_seqs = set(rid.rsplit(SEPARATOR, 1)[0] for rid in sample_ids)
    
    key_to_sub = {}
    for key in test_manifest:
        for cand in [key.replace("/", "_"), key.split("/")[-1], key, key.replace("/", "-")]:
            if cand in all_sub_seqs:
                key_to_sub[key] = cand
                break
        if key not in key_to_sub:
            kl = key.replace("/", "_").lower()
            for ss in all_sub_seqs:
                if ss.lower() == kl:
                    key_to_sub[key] = ss
                    break
    
    print(f"  Matched: {len(key_to_sub)}/{len(test_manifest)}")
    if len(key_to_sub) < len(test_manifest):
        unmatched = [k for k in test_manifest if k not in key_to_sub]
        print(f"  Unmatched: {unmatched[:5]}")
    
    # Frame count verification
    print(f"\n  Frame count verification:")
    for key in list(key_to_sub.keys())[:5]:
        sub_name = key_to_sub[key]
        m_frames = test_manifest[key]["n_frames"]
        s_frames = sum(1 for rid in sample_ids if rid.rsplit(SEPARATOR,1)[0] == sub_name)
        first_frame = min(int(rid.rsplit(SEPARATOR,1)[1]) for rid in sample_ids 
                         if rid.rsplit(SEPARATOR,1)[0] == sub_name)
        last_frame = max(int(rid.rsplit(SEPARATOR,1)[1]) for rid in sample_ids 
                        if rid.rsplit(SEPARATOR,1)[0] == sub_name)
        print(f"    {key}: manifest={m_frames}, sub={s_frames}, "
              f"frames={first_frame}..{last_frame}")

# ══════════════════════════════════════════════════════════════
# EXPLORE I: OD Predictions format
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE I: OD Predictions")
print("=" * 70)

train_od_dir = os.path.join(OD_PRED_DIR, "train")
test_od_dir = os.path.join(OD_PRED_DIR, "test")

print(f"Train OD dir exists: {os.path.exists(train_od_dir)}")
print(f"Test OD dir exists:  {os.path.exists(test_od_dir)}")

if os.path.exists(train_od_dir):
    train_od_files = sorted(glob.glob(f"{train_od_dir}/*.txt"))
    print(f"Train OD files: {len(train_od_files)}")
    print(f"  Samples: {[os.path.basename(f) for f in train_od_files[:5]]}")
    
    # Read first file to see format
    if train_od_files:
        with open(train_od_files[0]) as f:
            lines = f.readlines()
        print(f"\n  First file: {os.path.basename(train_od_files[0])}")
        print(f"  Lines: {len(lines)}")
        print(f"  First 3 lines: {[l.strip() for l in lines[:3]]}")

if os.path.exists(test_od_dir):
    test_od_files = sorted(glob.glob(f"{test_od_dir}/*.txt"))
    print(f"\nTest OD files: {len(test_od_files)}")
    print(f"  Samples: {[os.path.basename(f) for f in test_od_files[:5]]}")

# Check which train manifest keys have OD predictions
od_available = 0
for key in train_manifest:
    od_name = key.replace("/", "_") + ".txt"
    if os.path.exists(os.path.join(train_od_dir, od_name)):
        od_available += 1
print(f"\nTrain sequences with OD predictions: {od_available}/{len(train_manifest)}")

# ══════════════════════════════════════════════════════════════
# EXPLORE J: Build model + test forward pass
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE J: Model build + forward pass test")
print("=" * 70)

import yaml, importlib

# DotDict for config
class DotDict(dict):
    __getattr__ = dict.__getitem__
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__
    @staticmethod
    def from_nested(d):
        if isinstance(d, dict):
            return DotDict({k: DotDict.from_nested(v) for k, v in d.items()})
        if isinstance(d, list):
            return [DotDict.from_nested(i) for i in d]
        return d

# Load YAML
yaml_path = glob.glob(f"{REPO_DIR}/experiments/lightfc/{CONFIG_NAME}*.yaml")[0]
with open(yaml_path) as f:
    raw_cfg = yaml.safe_load(f)
cfg = DotDict.from_nested(raw_cfg)
print(f"Config loaded from {yaml_path}")
print(f"  TEMPLATE: {cfg.DATA.TEMPLATE.SIZE}px, factor={cfg.DATA.TEMPLATE.FACTOR}")
print(f"  SEARCH:   {cfg.DATA.SEARCH.SIZE}px, factor={cfg.DATA.SEARCH.FACTOR}")
print(f"  STRIDE:   {cfg.MODEL.BACKBONE.STRIDE}")

# Patch load_pretrain
try:
    pretrain_mod = importlib.import_module("lib.models.component.pretrain")
    pretrain_mod.load_pretrain = lambda model, path=None, **kw: model
except:
    for pyf in glob.glob(f"{REPO_DIR}/lib/models/**/*.py", recursive=True):
        with open(pyf) as f:
            txt = f.read()
        if "def load_pretrain" in txt:
            mod_path = pyf.replace(REPO_DIR+"/","").replace("/",".").replace(".py","")
            try:
                mod = importlib.import_module(mod_path)
                mod.load_pretrain = lambda model, path=None, **kw: model
            except: pass
            break

# Also patch in tracker_model.py directly
from lib.utils import load as load_module
load_module.load_pretrain = lambda model, path=None, **kw: model

# Build model
from lib.models.tracker_model import LightFC
model = LightFC(cfg, env_num=0, training=False)
print(f"✅ Model built")

# Load weights
ckpt_data = torch.load(CKPT_PATH, map_location='cpu')
state = ckpt_data.get('net', ckpt_data)
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"✅ Weights loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}")
if missing: print(f"  Missing: {missing[:5]}")
if unexpected: print(f"  Unexpected: {unexpected[:5]}")
del ckpt_data, state

# Force all to CUDA
def force_cuda(module):
    for name, buf in module.named_buffers(recurse=False):
        if buf is not None and not buf.is_cuda:
            module._buffers[name] = buf.cuda()
    for attr_name in list(module.__dict__.keys()):
        attr = getattr(module, attr_name, None)
        if isinstance(attr, torch.Tensor) and not attr.is_cuda:
            setattr(module, attr_name, attr.cuda())
    for child in module.children():
        force_cuda(child)

model.eval()
model.cuda()
force_cuda(model)

# Count params
total_p = sum(p.numel() for p in model.parameters())
backbone_p = sum(p.numel() for n, p in model.named_parameters() if 'backbone' in n)
fusion_p = sum(p.numel() for n, p in model.named_parameters() if 'fusion' in n)
head_p = sum(p.numel() for n, p in model.named_parameters() if 'head' in n)
print(f"  Params: total={total_p/1e6:.2f}M, backbone={backbone_p/1e6:.2f}M, "
      f"fusion={fusion_p/1e6:.2f}M, head={head_p/1e6:.2f}M")

# Count switch_to_deploy modules BEFORE deploying
n_deploy_backbone = sum(1 for m in model.backbone.modules() if hasattr(m, 'switch_to_deploy'))
n_deploy_head = sum(1 for m in model.head.modules() if hasattr(m, 'switch_to_deploy'))
print(f"  switch_to_deploy modules: backbone={n_deploy_backbone}, head={n_deploy_head}")

# Keys before deploy
keys_before_deploy = list(model.state_dict().keys())

# Deploy!
for m in model.backbone.modules():
    if hasattr(m, 'switch_to_deploy'):
        m.switch_to_deploy()
for m in model.head.modules():
    if hasattr(m, 'switch_to_deploy'):
        m.switch_to_deploy()
force_cuda(model)

keys_after_deploy = list(model.state_dict().keys())
print(f"  Keys before deploy: {len(keys_before_deploy)}")
print(f"  Keys after deploy:  {len(keys_after_deploy)}")
new_keys = set(keys_after_deploy) - set(keys_before_deploy)
lost_keys = set(keys_before_deploy) - set(keys_after_deploy)
if new_keys: print(f"  New keys: {sorted(new_keys)[:5]}")
if lost_keys: print(f"  Lost keys: {sorted(lost_keys)[:5]}")

# Forward pass test with real data
first_key = list(test_manifest.keys())[0]
info = test_manifest[first_key]
video_path = os.path.join(DATA_ROOT, info["video_path"])
annot_path = os.path.join(DATA_ROOT, info["annotation_path"])

cap = cv2.VideoCapture(video_path)
ret, frame_bgr = cap.read()
cap.release()
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

with open(annot_path) as f:
    parts = f.readline().strip().replace(',', ' ').split()
    init_bbox = [float(p) for p in parts[:4]]

print(f"\n  Test sequence: {first_key}")
print(f"  Image: {frame_rgb.shape}, Init bbox: {init_bbox}")

# Crop + preprocess
preprocessor = Preprocessor()
t_crop, t_rf, t_mask = repo_sample_target(frame_rgb, init_bbox, 2.0, output_sz=128)
s_crop, s_rf, s_mask = repo_sample_target(frame_rgb, init_bbox, 4.0, output_sz=256)

print(f"\n  sample_target template: crop={t_crop.shape} dtype={t_crop.dtype} rf={t_rf:.4f}")
print(f"  sample_target search:   crop={s_crop.shape} dtype={s_crop.dtype} rf={s_rf:.4f}")

t_proc = preprocessor.process(t_crop, t_mask)
s_proc = preprocessor.process(s_crop, s_mask)

print(f"  Preprocessed template: {t_proc.tensors.shape} device={t_proc.tensors.device} "
      f"range=[{t_proc.tensors.min():.3f}, {t_proc.tensors.max():.3f}]")
print(f"  Preprocessed search:   {s_proc.tensors.shape} device={s_proc.tensors.device}")

with torch.no_grad():
    z_feat = model.forward_backbone(t_proc.tensors.cuda())
    out = model.forward_tracking(z_feat, s_proc.tensors.cuda())

print(f"\n  z_feat: type={type(z_feat)}, "
      f"{'shape='+str(z_feat.shape) if isinstance(z_feat, torch.Tensor) else ''}")
print(f"\n  Model outputs:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"    '{k}': shape={v.shape}, range=[{v.min():.4f}, {v.max():.4f}], mean={v.mean():.4f}")

# Test cal_bbox
feat_sz = 256 // cfg.MODEL.BACKBONE.STRIDE
output_window = hann2d(torch.tensor([feat_sz, feat_sz]).long(), centered=True).cuda()
print(f"\n  hann2d: shape={output_window.shape}, range=[{output_window.min():.4f}, {output_window.max():.4f}]")

with torch.no_grad():
    response = output_window * out['score_map']
    cal_result = model.head.cal_bbox(response, out['size_map'], out['offset_map'])
    
    print(f"\n  response: shape={response.shape}")
    print(f"  cal_bbox output: shape={cal_result.shape}, "
          f"range=[{cal_result.min():.4f}, {cal_result.max():.4f}]")
    
    pred_boxes = cal_result.view(-1, 4)
    pred_mean = pred_boxes.mean(dim=0)
    print(f"  cal_bbox.view(-1,4): shape={pred_boxes.shape}")
    print(f"  mean across boxes: {pred_mean.cpu().tolist()}")
    
    # compute_box: scale to image pixels
    scaled = (pred_mean * 256 / s_rf).tolist()
    print(f"\n  compute_box scaled (* 256/{s_rf:.4f}): {[f'{v:.2f}' for v in scaled]}")
    
    # map_box_back
    cx_prev = init_bbox[0] + 0.5 * init_bbox[2]
    cy_prev = init_bbox[1] + 0.5 * init_bbox[3]
    half_side = 0.5 * 256 / s_rf
    cx_real = scaled[0] + (cx_prev - half_side)
    cy_real = scaled[1] + (cy_prev - half_side)
    mapped = [cx_real - 0.5*scaled[2], cy_real - 0.5*scaled[3], scaled[2], scaled[3]]
    
    H, W = frame_rgb.shape[:2]
    clipped = clip_box(mapped, H, W, margin=2)
    
    print(f"  map_box_back: {[f'{v:.2f}' for v in mapped]}")
    print(f"  clip_box:     {[f'{v:.2f}' for v in clipped]}")
    print(f"  Init bbox:    {init_bbox}")
    print(f"  Match? center_diff=({abs(clipped[0]+clipped[2]/2 - cx_prev):.1f}, "
          f"{abs(clipped[1]+clipped[3]/2 - cy_prev):.1f})")

# ══════════════════════════════════════════════════════════════
# EXPLORE K: End-to-end 10-frame tracking test
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("EXPLORE K: End-to-end tracking test (10 frames)")
print("=" * 70)

# Load GT
gt_bboxes = []
with open(annot_path) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.replace(',', ' ').split()
        if len(parts) >= 4:
            gt_bboxes.append([float(p) for p in parts[:4]])

print(f"GT frames: {len(gt_bboxes)}")
print(f"GT[0:5]: {gt_bboxes[:5]}")

# Track
state_track = list(init_bbox)
cap = cv2.VideoCapture(video_path)
preds = []

with torch.no_grad():
    for fidx in range(min(10, info['n_frames'])):
        ret, frame_bgr = cap.read()
        if not ret: break
        image = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        H, W, _ = image.shape
        
        if fidx == 0:
            z_patch, z_rf, z_mask = repo_sample_target(image, init_bbox, 2.0, output_sz=128)
            z_proc = preprocessor.process(z_patch, z_mask)
            z_feat = model.forward_backbone(z_proc.tensors.cuda())
            preds.append(list(init_bbox))
            state_track = list(init_bbox)
        else:
            x_patch, x_rf, x_mask = repo_sample_target(image, state_track, 4.0, output_sz=256)
            x_proc = preprocessor.process(x_patch, x_mask)
            out_dict = model.forward_tracking(z_feat=z_feat, x=x_proc.tensors.cuda())
            
            response = output_window * out_dict['score_map']
            pred_raw = model.head.cal_bbox(response, out_dict['size_map'], out_dict['offset_map'])
            pred_boxes = pred_raw.view(-1, 4)
            pred_mean = (pred_boxes.mean(dim=0) * 256 / x_rf).tolist()
            
            cx_prev = state_track[0] + 0.5 * state_track[2]
            cy_prev = state_track[1] + 0.5 * state_track[3]
            half_side = 0.5 * 256 / x_rf
            cx_real = pred_mean[0] + (cx_prev - half_side)
            cy_real = pred_mean[1] + (cy_prev - half_side)
            mapped = [cx_real - 0.5*pred_mean[2], cy_real - 0.5*pred_mean[3],
                      pred_mean[2], pred_mean[3]]
            
            state_track = clip_box(mapped, H, W, margin=2)
            preds.append(list(state_track))

cap.release()

# Compute IoU
def compute_iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[0]+a[2], b[0]+b[2]); y2 = min(a[1]+a[3], b[1]+b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter / max(union, 1e-7)

print(f"\nFrame-by-frame results:")
print(f"{'Frame':>5} | {'Pred x,y,w,h':>40} | {'GT x,y,w,h':>40} | {'IoU':>6} | {'Area':>8}")
print("-" * 115)
ious = []
for i in range(len(preds)):
    gt = gt_bboxes[i] if i < len(gt_bboxes) else [0,0,0,0]
    iou = compute_iou(preds[i], gt) if gt[2] > 0 and gt[3] > 0 else float('nan')
    if not math.isnan(iou): ious.append(iou)
    area = preds[i][2] * preds[i][3]
    init_area = init_bbox[2] * init_bbox[3]
    print(f"{i:5d} | [{preds[i][0]:8.1f},{preds[i][1]:8.1f},{preds[i][2]:8.1f},{preds[i][3]:8.1f}] "
          f"| [{gt[0]:8.1f},{gt[1]:8.1f},{gt[2]:8.1f},{gt[3]:8.1f}] "
          f"| {iou:6.3f} | {area:8.0f} {'⚠️' if area > init_area*3 else ''}")

if ious:
    print(f"\nMean IoU: {np.mean(ious):.4f} ({len(ious)} valid frames)")
else:
    print(f"\nNo valid GT frames found (all zeros)")
    print(f"Bbox areas: {[f'{p[2]*p[3]:.0f}' for p in preds]}")
    print(f"Areas stable? Max/Min ratio: "
          f"{max(p[2]*p[3] for p in preds[1:])/max(min(p[2]*p[3] for p in preds[1:]),1):.2f}"
          if len(preds) > 1 else "")

# ══════════════════════════════════════════════════════════════
# FINAL REPORT
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("FINAL REPORT — ALL DISCOVERIES")
print("=" * 70)
print(f"""
1. ENVIRONMENT: ✅ Repo cloned, patched, deps installed
2. CHECKPOINT:  ✅ {os.path.basename(CKPT_PATH)}
3. MODEL:       ✅ {total_p/1e6:.1f}M params (backbone={backbone_p/1e6:.1f}M + fusion={fusion_p/1e6:.1f}M + head={head_p/1e6:.1f}M)
4. DEPLOY:      {n_deploy_backbone + n_deploy_head} modules switched to deploy mode
5. CONFIG:      template={cfg.DATA.TEMPLATE.SIZE}px f={cfg.DATA.TEMPLATE.FACTOR}, search={cfg.DATA.SEARCH.SIZE}px f={cfg.DATA.SEARCH.FACTOR}

6. SAMPLE_TARGET: from {repo_sample_target.__module__}
   Returns: (crop_ndarray, resize_factor, mask_ndarray)
   
7. PREPROCESSOR: Converts (crop, mask) → NestedTensor on CUDA
   .tensors shape: (1, 3, H, W), normalized with ImageNet stats

8. MODEL OUTPUTS: pred_boxes(1,4), score_map(1,1,16,16), size_map(1,2,16,16), offset_map(1,2,16,16)

9. DECODE PIPELINE:
   response = hann2d * score_map
   cal_bbox(response, size_map, offset_map) → normalized bbox
   * 256 / resize_factor → pixel coords
   map_box_back → image coords
   clip_box → clamped

10. SUBMISSION: {len(all_rows)} rows, separator='{SEPARATOR if SEPARATOR else "UNKNOWN"}'
    Matched {len(key_to_sub) if 'key_to_sub' in dir() else '?'}/{len(test_manifest)} sequences
    
11. TRACKING TEST: Mean IoU = {np.mean(ious):.4f if ious else 'N/A (no valid GT)'}
    Area stability: {'STABLE' if len(preds)>1 and max(p[2]*p[3] for p in preds[1:])/max(min(p[2]*p[3] for p in preds[1:]),1) < 3 else 'CHECK OUTPUT ABOVE'}

12. OD PREDICTIONS: {od_available}/{len(train_manifest)} train sequences have teacher predictions
""")

if ious and np.mean(ious) > 0.5:
    print("✅ Pipeline VERIFIED — ready to write inference + training code")
elif not ious:
    print("⚠️ No GT available for first sequence — check area stability above")
    print("   If areas are stable, pipeline likely works")
else:
    print("❌ Pipeline has issues — check output above before proceeding")

✅ 1a: Repo ready: /kaggle/working/LightFC
✅ 1b: Dependencies installed
✅ 1c: Checkpoint: /kaggle/input/datasets/yussufothman/lightfc-vit-model/output/checkpoints/train/lightfc/baseline_v1_release_backbone_tinyvit/lightfc_ep0400.pth.tar (94.6MB)
✅ 1d: Patched 0 files
✅ 1e: Checkpoint linked

PART 1 COMPLETE — ENVIRONMENT READY

EXPLORE A: sample_target
Module: lib.train.data.processing_utils
File:   /kaggle/working/LightFC/lib/train/data/processing_utils.py
Signature: (im, target_bb, search_area_factor, output_sz=None, mask=None)

--- SOURCE CODE ---
def sample_target(im, target_bb, search_area_factor, output_sz=None, mask=None):
    """ Extracts a square crop centered at target_bb box, of area search_area_factor^2 times target_bb area

    args:
        im - cv image
        target_bb - target box [x, y, w, h]
        search_area_factor - Ratio of crop size to target size
        output_sz - (float) Size to which the extracted crop is resized (always square). If None, no resizing is do

ValueError: Invalid format specifier '.4f if ious else 'N/A (no valid GT)'' for object of type 'float'

In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 2: Baseline Inference — Original Weights
# ══════════════════════════════════════════════════════════════
# Uses EXACT native tracker pipeline:
#   sample_target → Preprocessor → forward_backbone/tracking →
#   hann2d * score_map → cal_bbox → compute_box → map_box_back → clip_box
#
# Expected Kaggle score: ~0.70 (baseline LightFC)
# Purpose: Verify pipeline before training
# ══════════════════════════════════════════════════════════════

import torch, os, csv, json, time
import numpy as np
import cv2

# Model is already loaded+deployed from exploration cell
# Verify
assert hasattr(model, 'head'), " Model not found — re-run Cell 1"
print(f"✅ Model ready, deployed={any('rbr_reparam' in k for k in model.state_dict())}")

@torch.no_grad()
def track_sequence(network, video_path, init_bbox, n_frames,
                   preprocessor, output_window, search_size=256,
                   template_size=128, search_factor=4.0, template_factor=2.0):
    """Exact replica of native lightFC tracker."""
    preds = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return [[0,0,0,0]] * n_frames

    state = list(init_bbox)
    z_feat = None

    for fidx in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            last = preds[-1] if preds else list(init_bbox)
            preds.extend([list(last)] * (n_frames - fidx))
            break

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        H, W, _ = image.shape

        if fidx == 0:
            # Initialize — exact copy of native tracker
            z_patch, _, z_mask = repo_sample_target(
                image, init_bbox, template_factor, output_sz=template_size)
            z_proc = preprocessor.process(z_patch, z_mask)
            z_feat = network.forward_backbone(z_proc.tensors)
            preds.append(list(init_bbox))
        else:
            # Track — exact copy of native tracker
            x_patch, x_rf, x_mask = repo_sample_target(
                image, state, search_factor, output_sz=search_size)
            x_proc = preprocessor.process(x_patch, x_mask)
            out_dict = network.forward_tracking(z_feat=z_feat, x=x_proc.tensors)

            # Decode: hann window × score_map → cal_bbox → scale → map_back → clip
            response = output_window * out_dict['score_map']
            pred_raw = network.head.cal_bbox(
                response, out_dict['size_map'], out_dict['offset_map'])
            pred_boxes = pred_raw.view(-1, 4)
            pred_scaled = (pred_boxes.mean(dim=0) * search_size / x_rf).tolist()

            # map_box_back
            cx_prev = state[0] + 0.5 * state[2]
            cy_prev = state[1] + 0.5 * state[3]
            half_side = 0.5 * search_size / x_rf
            cx_real = pred_scaled[0] + (cx_prev - half_side)
            cy_real = pred_scaled[1] + (cy_prev - half_side)
            mapped = [cx_real - 0.5*pred_scaled[2], cy_real - 0.5*pred_scaled[3],
                      pred_scaled[2], pred_scaled[3]]

            state = clip_box(mapped, H, W, margin=2)
            preds.append(list(state))

    cap.release()
    return preds

# ── Run on all test sequences ────────────────────────────────
print(f"Tracking {len(test_manifest)} test sequences...")
all_predictions = {}
t0 = time.time()

for seq_i, (key, info) in enumerate(test_manifest.items()):
    video_path = os.path.join(DATA_ROOT, info["video_path"])
    annot_path = os.path.join(DATA_ROOT, info["annotation_path"])

    with open(annot_path) as f:
        parts = f.readline().strip().replace(',', ' ').split()
        init_bbox = [float(p) for p in parts[:4]]

    preds = track_sequence(
        model, video_path, init_bbox, info["n_frames"],
        preprocessor, output_window)
    all_predictions[key] = preds

    if (seq_i + 1) % 15 == 0 or seq_i == 0:
        elapsed = time.time() - t0
        total = sum(len(v) for v in all_predictions.values())
        print(f"  [{seq_i+1}/{len(test_manifest)}] {total:,} frames, "
              f"{elapsed:.0f}s, {total/max(elapsed,1):.0f} fps")

total_frames = sum(len(v) for v in all_predictions.values())
elapsed = time.time() - t0
print(f"\n✅ Tracked {total_frames:,} frames in {elapsed:.0f}s "
      f"({total_frames/max(elapsed,1):.0f} fps)")

# ── Build submission ─────────────────────────────────────────
SAMPLE_SUB_PATH = f"{DATA_ROOT}/metadata/sample_submission.csv"

with open(SAMPLE_SUB_PATH) as f:
    reader = csv.reader(f)
    header = next(reader)
    all_rows = [row for row in reader]

sample_ids = [r[0] for r in all_rows]

# Parse row IDs: rsplit('_', 1) → (seq_name, frame_idx)
# Build reverse lookup: seq_name → manifest_key
# From exploration: seq names in CSV match manifest keys directly
sub_to_key = {}
for key in test_manifest:
    sub_to_key[key] = key  # Direct match (both contain '/')

SUB_PATH = f"{WORK}/submission_baseline.csv"
matched = 0
with open(SUB_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row_id in sample_ids:
        seq, frame_str = row_id.rsplit('_', 1)
        frame_idx = int(frame_str)
        preds = all_predictions.get(seq, [])
        if 0 <= frame_idx < len(preds):
            x, y, w, h = preds[frame_idx]
            writer.writerow([row_id, f"{x:.2f}", f"{y:.2f}", f"{w:.2f}", f"{h:.2f}"])
            matched += 1
        elif preds:
            x, y, w, h = preds[-1]
            writer.writerow([row_id, f"{x:.2f}", f"{y:.2f}", f"{w:.2f}", f"{h:.2f}"])
            matched += 1
        else:
            writer.writerow([row_id, "0.00", "0.00", "0.00", "0.00"])

print(f"\n✅ Baseline submission: {SUB_PATH}")
print(f"   Matched: {matched}/{len(sample_ids)} ({100*matched/len(sample_ids):.1f}%)")
print(f"   Size: {os.path.getsize(SUB_PATH)/1e6:.2f} MB")

✅ Model ready, deployed=True
Tracking 89 test sequences...
  [1/89] 585 frames, 13s, 45 fps
  [15/89] 5,211 frames, 115s, 45 fps
  [30/89] 9,235 frames, 191s, 48 fps
  [45/89] 24,222 frames, 474s, 51 fps
  [60/89] 47,063 frames, 915s, 51 fps
  [75/89] 61,817 frames, 1186s, 52 fps

✅ Tracked 74,293 frames in 1415s (53 fps)

✅ Baseline submission: /kaggle/working/submission_baseline.csv
   Matched: 74293/74293 (100.0%)
   Size: 3.66 MB


In [19]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3: GENERATE TRAINING PAIRS WITH MODERATE SEARCH JITTER
# ══════════════════════════════════════════════════════════════════════
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │                    PURPOSE                                         │
# │                                                                    │
# │  Pre-extract (template, search, GT_label, teacher_label) pairs     │
# │  from 217 training sequences and save as .pt chunks on disk.       │
# │                                                                    │
# │  This is done ONCE, then Cells 4-5 load these pairs for training.  │
# │  Pre-extraction avoids re-reading videos every epoch.              │
# └─────────────────────────────────────────────────────────────────────┘
#
# ── WHY PRE-EXTRACTION MAKES TRAINING FAST (~1.5-2s/epoch) ─────────
#
#   WITHOUT pre-extraction (typical training):
#     Every epoch, every batch:
#       1. Open video file (cv2.VideoCapture)     ~5ms
#       2. Seek to frame (sequential read)        ~10-50ms per frame
#       3. Decode frame (h264/h265 decode)         ~5ms
#       4. sample_target crop + resize             ~2ms
#       5. Normalize to tensor                     ~1ms
#       6. Transfer to GPU                         ~1ms
#     Total per pair: ~25-60ms
#     For 15,000 pairs: ~6-15 minutes PER EPOCH
#     15 epochs × 10 min = 2.5 HOURS
#
#   WITH pre-extraction (our approach):
#     Cell 3 (ONCE): Read all videos, extract all crops, save as tensors
#       ~5-8 minutes one-time cost
#     Cell 4-5 (every epoch):
#       1. Load .pt chunk from disk (already tensors)  → ~0ms (cached)
#       2. Transfer batch to GPU                       → ~0.5ms
#       3. Forward + backward pass                     → ~3ms
#     Total per batch: ~3.5ms
#     For 980 batches: ~3.4 seconds PER EPOCH
#     15 epochs × 3.4s = ~50 seconds total training
#
#   SPEEDUP: ~100× faster per epoch!
#
#   WHY THIS WORKS:
#     - Pairs are stored as fp16 tensors (half memory)
#     - ChunkDataset in Cell 4 loads ALL chunks into RAM
#     - DataLoader with pin_memory=True → fast GPU transfer
#     - No video I/O during training at all
#
# ── THE JITTER PROBLEM AND SOLUTION ─────────────────────────────────
#
#   PROBLEM: During inference, search crop is centered on PREVIOUS
#   PREDICTION (which may be slightly wrong). Target is NOT always
#   at the exact center of the search crop.
#
#   v1 (no jitter, scored 0.6714):
#     Search crop always centered on GT bbox
#     → Target always at (0.5, 0.5) in normalized crop coords
#     → Model learned: "always predict center"
#     → During inference: target drifts off-center → model can't find it
#     → Score DROPPED below baseline (0.6714 < 0.7020)
#
#   v2 (jitter=3.0, pred≠cal_bbox):
#     Search crop centered on heavily jittered GT
#     → Too aggressive: 3× bbox size shift
#     → 70% of pairs filtered out (target outside crop)
#     → Only 213 batches (vs 735 without jitter)
#     → pred_boxes ≠ cal_bbox (hanning window fought the model)
#     → Did NOT submit
#
#   v3 (THIS, jitter=0.5, scored 0.7292 ✅):
#     Search crop centered on MODERATELY jittered GT
#     → 0.5× bbox size shift (e.g., 50px for 100px bbox)
#     → Target stays well within crop
#     → Model learns "target is NEAR center but not EXACTLY at center"
#     → 980 batches (enough data, most pairs survive filter)
#     → pred_boxes = cal_bbox ✅ (hanning window agrees)
#     → +0.027 over baseline!
#
# ── HOW PAIRS ARE GENERATED ─────────────────────────────────────────
#
#   For each of 217 training sequences:
#
#   Step 1: Pre-sample frame pairs
#     - Find valid frames (where GT bbox w>0 and h>0)
#     - Randomly pair template frame t with search frame s
#     - Constraint: |t - s| ≤ MAX_GAP=100 (temporal proximity)
#     - Generate PAIRS_PER_SEQ=80 pairs per sequence
#
#   Step 2: Read needed frames from video
#     - Only read frames that appear in our pairs (not all frames)
#     - Sequential read with cv2.VideoCapture (seek by reading)
#
#   Step 3: For each (template_frame_t, search_frame_s) pair:
#
#     TEMPLATE (no jitter — template should be perfect):
#       bbox_t = GT[t]
#       crop_t = sample_target(frame_t, bbox_t, factor=2.0, sz=128)
#       → Template always shows target centered and clean
#
#     SEARCH (WITH jitter — simulate imperfect previous prediction):
#       bbox_s_gt = GT[s]                          ← real target location
#       bbox_s_jit = jitter(bbox_s_gt, 0.5, 0.1)  ← shifted center ±0.5×size
#       crop_s = sample_target(frame_s, bbox_s_jit, factor=4.0, sz=256)
#       → Search crop NOT centered on target (simulates inference)
#       → Target appears off-center in the crop
#
#     LABELS (relative to jittered crop, not image):
#       gt_label = transform_image_to_crop(GT[s], jittered_bbox, rf, crop_sz)
#       od_label = transform_image_to_crop(OD[s], jittered_bbox, rf, crop_sz)
#       → Both converted to normalized (cx,cy,w,h) in 0-1 crop coords
#       → gt_label center ≠ 0.5 (because crop is jittered)
#       → This teaches model to FIND the target, not just predict center
#
#   Step 4: Filter invalid pairs
#     - GT center must be within (0.05, 0.95) — visible in crop
#     - GT width and height must be > 0
#     - Relaxed from (0.0, 1.0) to keep more pairs
#
#   Step 5: Save as .pt chunks (500 pairs per chunk)
#     - Templates: (500, 3, 128, 128) fp16
#     - Searches:  (500, 3, 256, 256) fp16
#     - GT labels: (500, 4) fp16
#     - OD labels: (500, 4) fp16
#
# ── CONNECTION TO REPO'S TRAINING PIPELINE ──────────────────────────
#
#   The original LightFC training (which we DON'T use directly):
#     lib/train/data/processing_utils.py → jittered_center_crop()
#       Uses sample_target + transform_image_to_crop
#       Config: SEARCH.CENTER_JITTER=3, SEARCH.SCALE_JITTER=0.25
#
#   We replicate this logic but with MODERATE jitter (0.5, 0.1):
#     jitter_bbox_for_search() ← our version of the jitter
#     sample_target()          ← repo's exact function
#     transform_image_to_crop() ← repo's exact function
#
#   The original trains for 400 epochs on millions of pairs from
#   LaSOT+GOT10K+COCO+TrackingNet with heavy jitter (3.0).
#   We can't afford that — 15 epochs on 15K pairs with gentle jitter.
#
# ── DATA SPLIT ──────────────────────────────────────────────────────
#
#   217 sequences with both GT and ODTrack predictions
#   → 90% train (~195 seqs) + 10% val (~22 seqs)
#   → ~15,600 train pairs + ~1,680 val pairs
#   → ~980 train batches + ~105 val batches (batch_size=16)
#
# ══════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
# CELL 3 (FIXED v2): Training Pairs with MODERATE Search Jitter
# ══════════════════════════════════════════════════════════════
#
# v1 (no jitter):     scored 0.6714 (model predicted "always center")
# v2 (jitter=3.0):    pred_boxes ≠ cal_bbox (too aggressive, lost 70% data)
# v3 (THIS, jitter=0.5): moderate — teaches off-center without chaos
# ══════════════════════════════════════════════════════════════

import torch, os, glob, time, gc, math
import numpy as np
import cv2

from lib.train.data.processing_utils import sample_target as repo_sample_target
from lib.train.data.processing_utils import transform_image_to_crop

PAIRS_DIR = f"{WORK}/train_pairs"
os.makedirs(PAIRS_DIR, exist_ok=True)
for old in glob.glob(f"{PAIRS_DIR}/*.pt"):
    os.remove(old)

# ── Config ───────────────────────────────────────────────────
TEMPLATE_SZ = 128
SEARCH_SZ = 256
TEMPLATE_F = 2.0
SEARCH_F = 4.0
PAIRS_PER_SEQ = 80             # ← increased from 60 to compensate for filtered pairs
MAX_GAP = 100
CHUNK_SIZE = 500
SEED = 42

# ══════════════════════════════════════════════════════════════
# KEY CHANGES: Moderate jitter
# ══════════════════════════════════════════════════════════════
SEARCH_CENTER_JITTER = 0.5    # was 3.0, now 0.5 (gentle shift)
SEARCH_SCALE_JITTER = 0.1     # was 0.25, now 0.1 (±10% scale)
TEMPLATE_CENTER_JITTER = 0.0
TEMPLATE_SCALE_JITTER = 0.0

np.random.seed(SEED)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def img_to_tensor_fp16(crop_uint8):
    t = crop_uint8.astype(np.float32) / 255.0
    t = (t - MEAN) / STD
    return torch.from_numpy(t.transpose(2, 0, 1)).half()


def jitter_bbox_for_search(bbox_xywh, center_jitter, scale_jitter):
    x, y, w, h = [float(v) for v in bbox_xywh]
    cx = x + w / 2
    cy = y + h / 2
    
    max_dim = max(w, h)
    
    # Center jitter
    cx += np.random.uniform(-center_jitter, center_jitter) * max_dim
    cy += np.random.uniform(-center_jitter, center_jitter) * max_dim
    
    # Scale jitter
    w *= np.exp(np.random.uniform(-scale_jitter, scale_jitter))
    h *= np.exp(np.random.uniform(-scale_jitter, scale_jitter))
    
    w = max(w, 1.0)
    h = max(h, 1.0)
    
    return [cx - w/2, cy - h/2, w, h]


def jitter_bbox_for_template(bbox_xywh, center_jitter=0.0, scale_jitter=0.0):
    if center_jitter == 0 and scale_jitter == 0:
        return list(bbox_xywh)
    return jitter_bbox_for_search(bbox_xywh, center_jitter, scale_jitter)


def load_bboxes(txt_path):
    bboxes = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.replace(',', ' ').split()
            if len(parts) >= 4:
                bboxes.append([float(parts[i]) for i in range(4)])
    return np.array(bboxes, dtype=np.float32) if bboxes else np.zeros((0,4), dtype=np.float32)


# ── Build sequence list ──────────────────────────────────────
train_sequences = []
for key, info in train_manifest.items():
    gt_path = os.path.join(DATA_ROOT, info["annotation_path"])
    od_name = key.replace("/", "_") + ".txt"
    od_path = os.path.join(OD_PRED_DIR, "train", od_name)
    video_path = os.path.join(DATA_ROOT, info["video_path"])
    
    if not all(os.path.exists(p) for p in [gt_path, od_path, video_path]):
        continue
    
    gt = load_bboxes(gt_path)
    od = load_bboxes(od_path)
    
    if len(gt) == 0 or len(od) == 0: continue
    n = min(len(gt), len(od), info["n_frames"])
    if n < 10: continue
    
    train_sequences.append({
        "key": key, "video_path": video_path,
        "gt": gt[:n], "od": od[:n], "n_frames": n,
    })

print(f"✅ Sequences with GT+OD: {len(train_sequences)}")

np.random.seed(SEED)
indices = np.random.permutation(len(train_sequences))
n_val = max(1, int(len(train_sequences) * 0.1))
val_seqs = [train_sequences[i] for i in sorted(indices[:n_val])]
trn_seqs = [train_sequences[i] for i in sorted(indices[n_val:])]
print(f"  Split: {len(trn_seqs)} train, {len(val_seqs)} val")


def presample_pairs(n_frames, gt, n_pairs, max_gap):
    valid = np.where((gt[:, 2] > 0) & (gt[:, 3] > 0))[0]
    if len(valid) < 2: return [], set()
    pairs, attempts = [], 0
    while len(pairs) < n_pairs and attempts < n_pairs * 5:
        attempts += 1
        t = np.random.choice(valid)
        lo, hi = max(valid[0], t - max_gap), min(valid[-1], t + max_gap)
        cands = valid[(valid >= lo) & (valid <= hi) & (valid != t)]
        if len(cands) == 0: continue
        s = np.random.choice(cands)
        pairs.append((int(t), int(s)))
    needed = set()
    for t, s in pairs: needed.add(t); needed.add(s)
    return pairs, needed


def read_needed_frames(video_path, needed):
    if not needed: return {}
    frames = {}
    max_idx = max(needed)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return {}
    idx = 0
    while idx <= max_idx:
        ret, frame = cap.read()
        if not ret: break
        if idx in needed:
            frames[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        idx += 1
    cap.release()
    return frames


def generate_pairs(sequences, split_name):
    all_t, all_s, all_g, all_o = [], [], [], []
    chunk_idx = 0
    total_pairs = 0
    t0 = time.time()
    
    for seq_i, seq in enumerate(sequences):
        gt = seq["gt"]
        od = seq["od"]
        
        pairs, needed = presample_pairs(seq["n_frames"], gt, PAIRS_PER_SEQ, MAX_GAP)
        if not pairs: continue
        
        frames = read_needed_frames(seq["video_path"], needed)
        if not frames: continue
        
        for t_idx, s_idx in pairs:
            if t_idx not in frames or s_idx not in frames: continue
            
            t_frame = frames[t_idx]
            s_frame = frames[s_idx]
            
            s_gt = gt[s_idx].tolist()
            s_od = od[s_idx].tolist()
            if s_gt[2] <= 0 or s_gt[3] <= 0: continue
            if s_od[2] <= 0 or s_od[3] <= 0: continue
            
            # ═══ TEMPLATE: crop centered on GT (no jitter) ═══
            t_gt = gt[t_idx].tolist()
            t_bbox = jitter_bbox_for_template(t_gt, TEMPLATE_CENTER_JITTER, TEMPLATE_SCALE_JITTER)
            try:
                t_crop, t_rf, t_mask = repo_sample_target(
                    t_frame, t_bbox, TEMPLATE_F, output_sz=TEMPLATE_SZ)
            except: continue
            
            # ═══ SEARCH: crop centered on JITTERED GT ═══
            s_jittered = jitter_bbox_for_search(
                s_gt, SEARCH_CENTER_JITTER, SEARCH_SCALE_JITTER)
            try:
                s_crop, s_rf, s_mask = repo_sample_target(
                    s_frame, s_jittered, SEARCH_F, output_sz=SEARCH_SZ)
            except: continue
            
            # ═══ LABELS: GT and OD relative to JITTERED crop ═══
            crop_sz_tensor = torch.tensor([SEARCH_SZ, SEARCH_SZ], dtype=torch.float32)
            
            gt_box_tensor = torch.tensor(s_gt, dtype=torch.float32)
            jit_box_tensor = torch.tensor(s_jittered, dtype=torch.float32)
            
            gt_transformed = transform_image_to_crop(
                gt_box_tensor, jit_box_tensor, s_rf, crop_sz_tensor, normalize=True)
            gt_x1, gt_y1, gt_w, gt_h = gt_transformed.tolist()
            gt_label = [gt_x1 + gt_w/2, gt_y1 + gt_h/2, gt_w, gt_h]
            
            od_box_tensor = torch.tensor(s_od, dtype=torch.float32)
            od_transformed = transform_image_to_crop(
                od_box_tensor, jit_box_tensor, s_rf, crop_sz_tensor, normalize=True)
            od_x1, od_y1, od_w, od_h = od_transformed.tolist()
            od_label = [od_x1 + od_w/2, od_y1 + od_h/2, od_w, od_h]
            
            # ═══ RELAXED FILTER (was 0.0-1.0, now 0.05-0.95) ═══
            if gt_label[2] <= 0 or gt_label[3] <= 0: continue
            if gt_label[0] < 0.05 or gt_label[0] > 0.95: continue  # ← relaxed
            if gt_label[1] < 0.05 or gt_label[1] > 0.95: continue  # ← relaxed
            
            all_t.append(img_to_tensor_fp16(t_crop))
            all_s.append(img_to_tensor_fp16(s_crop))
            all_g.append(torch.tensor(gt_label, dtype=torch.float16))
            all_o.append(torch.tensor(od_label, dtype=torch.float16))
            total_pairs += 1
            
            if len(all_t) >= CHUNK_SIZE:
                torch.save({
                    'templates': torch.stack(all_t),
                    'searches':  torch.stack(all_s),
                    'gt_labels': torch.stack(all_g),
                    'od_labels': torch.stack(all_o),
                }, f"{PAIRS_DIR}/{split_name}_chunk{chunk_idx:04d}.pt")
                chunk_idx += 1
                all_t.clear(); all_s.clear()
                all_g.clear(); all_o.clear()
        
        del frames
        
        if (seq_i + 1) % 25 == 0:
            elapsed = time.time() - t0
            eta = (len(sequences) - seq_i - 1) / ((seq_i + 1) / elapsed)
            print(f"  [{split_name}] {seq_i+1}/{len(sequences)} | "
                  f"{total_pairs} pairs | {elapsed:.0f}s | ETA {eta:.0f}s")
    
    if all_t:
        torch.save({
            'templates': torch.stack(all_t),
            'searches':  torch.stack(all_s),
            'gt_labels': torch.stack(all_g),
            'od_labels': torch.stack(all_o),
        }, f"{PAIRS_DIR}/{split_name}_chunk{chunk_idx:04d}.pt")
        chunk_idx += 1
    
    elapsed = time.time() - t0
    print(f"✅ [{split_name}] {total_pairs} pairs, {chunk_idx} chunks, {elapsed:.0f}s")
    return total_pairs


# ── Sanity check: verify labels vary but stay reasonable ─────
test_seq = train_sequences[0]
test_frames = read_needed_frames(test_seq["video_path"], {0, 1, 2, 3, 4})
centers = []
for fidx in [0, 1, 2, 3, 4]:
    if fidx not in test_frames: continue
    gt_bbox = test_seq["gt"][fidx].tolist()
    if gt_bbox[2] <= 0: continue
    
    for trial in range(5):
        jit = jitter_bbox_for_search(gt_bbox, SEARCH_CENTER_JITTER, SEARCH_SCALE_JITTER)
        try:
            _, rf, _ = repo_sample_target(test_frames[fidx], jit, SEARCH_F, output_sz=SEARCH_SZ)
        except: continue
        crop_sz_t = torch.tensor([SEARCH_SZ, SEARCH_SZ], dtype=torch.float32)
        gt_t = torch.tensor(gt_bbox, dtype=torch.float32)
        jit_t = torch.tensor(jit, dtype=torch.float32)
        transformed = transform_image_to_crop(gt_t, jit_t, rf, crop_sz_t, normalize=True)
        x1, y1, w, h = transformed.tolist()
        cx, cy = x1 + w/2, y1 + h/2
        centers.append((cx, cy))

del test_frames
print(f"\n  Sanity check — GT centers in jittered crops (should vary MODERATELY):")
for i, (cx, cy) in enumerate(centers[:10]):
    offset = max(abs(cx-0.5), abs(cy-0.5))
    status = '✅ shifted' if offset > 0.02 else '(near center)'
    print(f"    Trial {i}: cx={cx:.3f}, cy={cy:.3f} {status}")

cx_std = np.std([c[0] for c in centers])
cy_std = np.std([c[1] for c in centers])
print(f"  Center std: cx={cx_std:.4f}, cy={cy_std:.4f}")
if cx_std > 0.02 and cx_std < 0.15:
    print(f"  ✅ Good moderate variation!")
elif cx_std >= 0.15:
    print(f"  ⚠️  High variation — may still work but watch pred_boxes vs cal_bbox")
else:
    print(f"  ❌ Too little variation")


# ── Generate ─────────────────────────────────────────────────
print(f"\nGenerating TRAIN pairs...")
n_trn = generate_pairs(trn_seqs, "train")
print(f"\nGenerating VAL pairs...")
n_val = generate_pairs(val_seqs, "val")

gc.collect()
disk_mb = sum(os.path.getsize(f) for f in glob.glob(f"{PAIRS_DIR}/*.pt")) / 1e6
print(f"\n✅ Cell 3 complete: train={n_trn}, val={n_val}, disk={disk_mb:.0f}MB")

# ── CRITICAL CHECK: batch count ──────────────────────────────
expected_train_batches = n_trn // 16
print(f"   Expected train batches: ~{expected_train_batches} (need 500+ for good training)")
if expected_train_batches < 400:
    print(f"   ⚠️  Low batch count. Consider increasing PAIRS_PER_SEQ to 100")

✅ Sequences with GT+OD: 217
  Split: 196 train, 21 val

  Sanity check — GT centers in jittered crops (should vary MODERATELY):
    Trial 0: cx=0.535, cy=0.383 ✅ shifted
    Trial 1: cx=0.495, cy=0.428 ✅ shifted
    Trial 2: cx=0.598, cy=0.423 ✅ shifted
    Trial 3: cx=0.646, cy=0.337 ✅ shifted
    Trial 4: cx=0.375, cy=0.434 ✅ shifted
    Trial 5: cx=0.482, cy=0.423 ✅ shifted
    Trial 6: cx=0.327, cy=0.492 ✅ shifted
    Trial 7: cx=0.330, cy=0.477 ✅ shifted
    Trial 8: cx=0.573, cy=0.430 ✅ shifted
    Trial 9: cx=0.530, cy=0.368 ✅ shifted
  Center std: cx=0.1110, cy=0.0737
  ✅ Good moderate variation!

Generating TRAIN pairs...
  [train] 25/196 | 2000 pairs | 97s | ETA 662s
  [train] 50/196 | 4000 pairs | 125s | ETA 365s
  [train] 75/196 | 6000 pairs | 238s | ETA 384s
  [train] 100/196 | 8000 pairs | 369s | ETA 354s
  [train] 125/196 | 10000 pairs | 483s | ETA 274s
  [train] 150/196 | 12000 pairs | 676s | ETA 207s
  [train] 175/196 | 14000 pairs | 760s | ETA 91s
✅ [train] 15680 pair

In [20]:
# ══════════════════════════════════════════════════════════════
# CELL 4: Dataset and DataLoader
# ══════════════════════════════════════════════════════════════

import torch
from torch.utils.data import Dataset, DataLoader
import glob, gc

class ChunkDataset(Dataset):
    def __init__(self, pairs_dir, split_name):
        chunks = sorted(glob.glob(f"{pairs_dir}/{split_name}_chunk*.pt"))
        assert chunks, f"❌ No chunks for {split_name}"
        
        t_list, s_list, g_list, o_list = [], [], [], []
        for cf in chunks:
            d = torch.load(cf, map_location='cpu')
            t_list.append(d['templates'])
            s_list.append(d['searches'])
            g_list.append(d['gt_labels'])
            o_list.append(d['od_labels'])
        
        self.templates = torch.cat(t_list)
        self.searches  = torch.cat(s_list)
        self.gt_labels = torch.cat(g_list)
        self.od_labels = torch.cat(o_list)
        print(f"  [{split_name}] {len(self)} pairs loaded")
    
    def __len__(self): return len(self.templates)
    
    def __getitem__(self, idx):
        return (self.templates[idx].float(),
                self.searches[idx].float(),
                self.gt_labels[idx].float(),
                self.od_labels[idx].float())

train_dataset = ChunkDataset(PAIRS_DIR, "train")
val_dataset   = ChunkDataset(PAIRS_DIR, "val")
gc.collect()

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2,
                        pin_memory=True, drop_last=False)

print(f"\n✅ Cell 4: train={len(train_loader)} batches, val={len(val_loader)} batches")

  [train] 15680 pairs loaded
  [val] 1680 pairs loaded

✅ Cell 4: train=980 batches, val=105 batches


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: STAGE 1 TRAINING — GT + TEACHER DISTILLATION
# ══════════════════════════════════════════════════════════════════════
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │                    PURPOSE                                        │
# │                                                                   │
# │  Fine-tune the LightFC head using a distillation loss:            │
# │    Loss = 0.5 × (GIoU + L1)(pred, GT) + 0.5 × SmoothL1(pred, OD)  │
# │                                                                   │
# │  Only the HEAD is trained (4.81M params).                         │
# │  Backbone (2.44M) and Fusion (0.03M) are FROZEN.                  │
# │                                                                    │
# │  Score progression:                                                │
# │    Baseline (no training): 0.7020                                  │
# │    After Stage 1:          0.7292 (+0.027)                         │
# └─────────────────────────────────────────────────────────────────────┘
#
# ── CRITICAL: REPVGG TRAIN vs DEPLOY ───────────────────────────────
#
#   The head uses RepVGG blocks. These have TWO modes:
#
#   TRAINING MODE (what we need here):
#     Input ──┬── rbr_dense (3×3 conv + BatchNorm) ──┬── Add → Output
#             └── rbr_3x3   (3×3 conv + BatchNorm) ──┘
#     • Two parallel paths → richer gradients → better learning
#     • BatchNorm tracks running mean/var (needed for training)
#     • Checkpoint keys: rbr_dense.conv.weight, rbr_3x3.bn.bias, etc.
#
#   DEPLOY MODE (Cell 1 already switched to this — we must UNDO it):
#     Input ──── rbr_reparam (single 3×3 conv + bias) ──── Output
#     • Mathematically identical output but 2× faster
#     • BatchNorm is GONE (fused into conv weights)
#     • CANNOT train: no BN stats, no parallel paths for gradients
#     • Checkpoint keys: rbr_reparam.weight, rbr_reparam.bias
#
#   SOLUTION: Step 5a reloads the ORIGINAL checkpoint from disk.
#   This gives us back rbr_dense + rbr_3x3 (training mode).
#   We verify: has_dense=True, has_reparam=False before proceeding.
#
# ── WHAT'S FROZEN AND WHAT'S TRAINED ───────────────────────────────
#
#   FROZEN (backbone + fusion):
#     • backbone.eval() — BN running stats don't update
#     • fusion.eval()   — BN running stats don't update
#     • All params: requires_grad = False
#     • These are pretrained on ImageNet/tracking datasets
#     • Modifying them risks destroying learned representations
#     • 2.44M + 0.03M = 2.47M frozen params
#
#   TRAINED (head only):
#     • head.train() — BN running stats update during training
#     • All params: requires_grad = True
#     • Score/size/offset prediction branches
#     • 4.81M trainable params
#     • This is where tracking "decisions" are made
#
# ── LOSS FUNCTION EXPLAINED ─────────────────────────────────────────
#
#   total_loss = α × gt_loss + (1-α) × teacher_loss
#
#   gt_loss = GIoU_loss(pred, GT) + 2.0 × L1_loss(pred, GT)
#     • GIoU: measures overlap between predicted and true bbox
#       - IoU but also penalizes non-overlapping distance
#       - Range: -1 (worst) to 1 (perfect), loss = 1 - GIoU
#     • L1: measures absolute coordinate difference
#       - Helps with fast convergence on position
#     • Weight 2.0 (reduced from initial 5.0 to let GIoU dominate)
#
#   teacher_loss = SmoothL1_loss(pred, ODTrack_prediction)
#     • Softer than L1 (quadratic near zero, linear far)
#     • Teaches student to mimic ODTrack's behavior
#     • ODTrack scored 0.8020 — strong teacher signal
#
#   α = 0.5 (equal weight):
#     • GT provides correctness (where target really is)
#     • Teacher provides tracker-like behavior patterns
#     • Equal balance found to work best empirically
#
# ── HOW model.forward() WORKS DURING TRAINING ──────────────────────
#
#   From tracker_model.py:
#     def forward(self, z, x):
#         if self.training:        ← THIS PATH
#             z = self.backbone(z)   # template through backbone
#             x = self.backbone(x)   # search through backbone
#             opt = self.fusion(z, x) # correlate features
#             out = self.head(opt)    # predict score/size/offset
#         return out
#
#   Output: {'pred_boxes': (B,4), 'score_map': (B,1,16,16), ...}
#
#   We supervise pred_boxes directly.
#   pred_boxes IS the output of cal_bbox internally (verified:
#   pred_boxes == cal_bbox gives difference = [0.0, 0.0, 0.0, 0.0])
#
#   So training pred_boxes = training the score_map + size_map +
#   offset_map that cal_bbox reads from during inference.
#
# ── BUT WAIT: model.training vs backbone.eval() ────────────────────
#
#   We set:
#     model.train()        → model.training = True
#     model.backbone.eval() → backbone BN in eval mode
#     model.fusion.eval()   → fusion BN in eval mode
#     (head stays in train mode)
#
#   model.training=True routes forward() to the training path
#   (z through backbone, x through backbone, fusion, head).
#
#   backbone.eval() means backbone's BatchNorm layers use their
#   stored running_mean/running_var (don't update from batch stats).
#   This is correct because we FROZE the backbone.
#
#   head stays in train() mode so its BatchNorm layers DO update
#   running stats from batch data. This is correct because we're
#   TRAINING the head.
#
# ── HYPERPARAMETERS ─────────────────────────────────────────────────
#
#   EPOCHS = 15
#     • Enough for convergence (val IoU plateaus by epoch 12-14)
#     • Not so many as to overfit on 217 sequences
#
#   LR = 5e-5 (with warmup → cosine decay)
#     • Low enough to preserve pretrained head knowledge
#     • Higher LR (1e-4) was tried → disrupted pretrained weights
#     • Warmup 3 epochs: LR ramps from ~1.7e-5 to 5e-5
#     • Cosine decay: smoothly decreases to 1e-6
#
#   BATCH_SIZE = 16
#     • Fits in GPU memory with mixed precision (autocast)
#
#   GRAD_CLIP = 1.0
#     • Prevents gradient explosion from rare bad pairs
#
#   Mixed precision (autocast + GradScaler):
#     • Forward pass in fp16 (faster, less memory)
#     • Gradients scaled to prevent underflow
#     • ~2× speedup over fp32
#
# ── TRAINING RESULTS ────────────────────────────────────────────────
#
#   Epoch  1: Val IoU = 0.8117 (model already decent from pretrained)
#   Epoch  5: Val IoU = 0.8260 (fast learning phase)
#   Epoch 14: Val IoU = 0.8287 ⭐ (best)
#   Epoch 15: Val IoU = 0.8286 (slight decline → correctly stopped)
#
#   Train-Val gap: 0.849 - 0.829 = 0.020 (healthy, minimal overfit)
#   Improvement:   +0.017 over pretrained crop-level IoU
#
#   Kaggle score: 0.7292 (+0.027 over baseline 0.7020)
#
# ══════════════════════════════════════════════════════════════
# CELL 5: Stage 1 Training — GT + Teacher Distillation
# ══════════════════════════════════════════════════════════════
#
# CRITICAL: Must reload ORIGINAL (non-deployed) weights!
# switch_to_deploy deletes rbr_dense/rbr_3x3 branches.
# Training needs those branches (they have gradients).
# Deploy is for INFERENCE ONLY.
# ══════════════════════════════════════════════════════════════

import torch, time, os, math
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

SAVE_DIR = f"{WORK}/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 5a: Reload original weights (undo deploy) ───────────────
ckpt_data = torch.load(CKPT_PATH, map_location='cpu')
state_orig = ckpt_data.get('net', ckpt_data)

# Rebuild model from scratch (deploy destroyed branches)
from lib.models.tracker_model import LightFC
model = LightFC(cfg, env_num=0, training=False)
missing, unexpected = model.load_state_dict(state_orig, strict=False)
print(f"✅ Reloaded original weights (pre-deploy). Missing={len(missing)}, Unexpected={len(unexpected)}")
del ckpt_data, state_orig

# Verify: must have rbr_dense keys (training mode RepVGG)
has_dense = any('rbr_dense' in k for k in model.state_dict())
has_reparam = any('rbr_reparam' in k for k in model.state_dict())
assert has_dense and not has_reparam, \
    f"❌ Wrong mode: dense={has_dense}, reparam={has_reparam}. Need training mode."
print(f"✅ RepVGG in training mode (rbr_dense present, rbr_reparam absent)")

model.cuda()
force_cuda(model)

# ── 5b: Freeze backbone + fusion, train only head ───────────
for name, param in model.named_parameters():
    param.requires_grad = 'head' in name

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"✅ Trainable: {trainable/1e6:.2f}M (head), Frozen: {frozen/1e6:.2f}M (backbone+fusion)")


# ── 5c: Loss functions ──────────────────────────────────────
def giou_loss(pred, target):
    """GIoU loss for (cx, cy, w, h) normalized."""
    # To x1y1x2y2
    p_x1 = pred[:, 0] - pred[:, 2] / 2
    p_y1 = pred[:, 1] - pred[:, 3] / 2
    p_x2 = pred[:, 0] + pred[:, 2] / 2
    p_y2 = pred[:, 1] + pred[:, 3] / 2
    
    t_x1 = target[:, 0] - target[:, 2] / 2
    t_y1 = target[:, 1] - target[:, 3] / 2
    t_x2 = target[:, 0] + target[:, 2] / 2
    t_y2 = target[:, 1] + target[:, 3] / 2
    
    inter_x1 = torch.max(p_x1, t_x1)
    inter_y1 = torch.max(p_y1, t_y1)
    inter_x2 = torch.min(p_x2, t_x2)
    inter_y2 = torch.min(p_y2, t_y2)
    inter = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    
    p_area = (p_x2 - p_x1).clamp(0) * (p_y2 - p_y1).clamp(0)
    t_area = (t_x2 - t_x1).clamp(0) * (t_y2 - t_y1).clamp(0)
    union = p_area + t_area - inter + 1e-7
    iou = inter / union
    
    enc_x1 = torch.min(p_x1, t_x1)
    enc_y1 = torch.min(p_y1, t_y1)
    enc_x2 = torch.max(p_x2, t_x2)
    enc_y2 = torch.max(p_y2, t_y2)
    enc_area = (enc_x2 - enc_x1).clamp(0) * (enc_y2 - enc_y1).clamp(0) + 1e-7
    
    giou = iou - (enc_area - union) / enc_area
    return (1 - giou).mean(), iou.detach().mean()


def compute_loss(pred, gt, od, alpha=0.7, l1_weight=5.0):
    """Stage 1: alpha * (GIoU + L1)(pred, GT) + (1-alpha) * SmoothL1(pred, teacher)"""
    giou_l, mean_iou = giou_loss(pred, gt)
    l1_l = F.l1_loss(pred, gt)
    gt_loss = giou_l + l1_weight * l1_l
    
    teacher_loss = F.smooth_l1_loss(pred, od)
    total = alpha * gt_loss + (1 - alpha) * teacher_loss
    return total, gt_loss.item(), teacher_loss.item(), mean_iou.item()


# ── 5d: Extract prediction from model output ────────────────
# pred_boxes from head is (B, 4) cxcywh normalized
# This is the DIRECT regression output (not from cal_bbox)
# During training, model.training=True → forward(z, x) → head(fusion(z,x))
# head.forward returns {'pred_boxes': ..., 'score_map': ..., ...}
# pred_boxes is what we supervise against

OUTPUT_KEY = 'pred_boxes'

def get_pred(output):
    pred = output[OUTPUT_KEY]
    if pred.dim() == 3: pred = pred[:, 0, :]
    return pred


# ── 5e: Hyperparameters ─────────────────────────────────────
EPOCHS = 15
LR = 5e-5          # Lower than before to preserve pretrained head
MIN_LR = 1e-6
WARMUP = 3          # Gentler warmup
ALPHA = 0.5         # Equal GT and teacher weight
L1_WEIGHT = 2.0     # Lower L1 (let GIoU dominate)
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return max(MIN_LR / LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler('cuda')


# ── 5f: Train + validate functions ──────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    # CRITICAL: model.training=True routes to forward(z_image, x_image)
    # which passes both through backbone. This is the TRAINING path.
    model.train()
    model.backbone.eval()   # Frozen BN
    model.fusion.eval()     # Frozen BN
    # head stays in train() mode
    
    loss_sum, iou_sum, n = 0, 0, 0
    for templates, searches, gt_labels, od_labels in loader:
        templates = templates.cuda(non_blocking=True)
        searches = searches.cuda(non_blocking=True)
        gt_labels = gt_labels.cuda(non_blocking=True)
        od_labels = od_labels.cuda(non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            output = model(templates, searches)
            pred = get_pred(output)
            loss, _, _, iou = compute_loss(pred, gt_labels, od_labels, ALPHA, L1_WEIGHT)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        loss_sum += loss.item()
        iou_sum += iou
        n += 1
    return loss_sum / max(n, 1), iou_sum / max(n, 1)


@torch.no_grad()
def validate(model, loader):
    model.train()           # Same forward path
    model.backbone.eval()
    model.fusion.eval()
    model.head.eval()       # But head BN doesn't update during val
    
    loss_sum, iou_sum, n = 0, 0, 0
    for templates, searches, gt_labels, od_labels in loader:
        templates = templates.cuda(non_blocking=True)
        searches = searches.cuda(non_blocking=True)
        gt_labels = gt_labels.cuda(non_blocking=True)
        od_labels = od_labels.cuda(non_blocking=True)
        
        with autocast('cuda'):
            output = model(templates, searches)
            pred = get_pred(output)
            loss, _, _, iou = compute_loss(pred, gt_labels, od_labels, ALPHA, L1_WEIGHT)
        
        loss_sum += loss.item()
        iou_sum += iou
        n += 1
    return loss_sum / max(n, 1), iou_sum / max(n, 1)


# ── 5g: Training loop ───────────────────────────────────────
print("=" * 60)
print("STAGE 1 TRAINING")
print(f"Epochs={EPOCHS}, LR={LR}, Alpha={ALPHA}, L1w={L1_WEIGHT}")
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")
print("=" * 60)

best_val_iou = 0
best_epoch = -1

for epoch in range(EPOCHS):
    t0 = time.time()
    
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_iou = validate(model, val_loader)
    scheduler.step()
    
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    
    print(f"Ep {epoch+1:2d}/{EPOCHS} | "
          f"Train: loss={train_loss:.4f} IoU={train_iou:.4f} | "
          f"Val: loss={val_loss:.4f} IoU={val_iou:.4f} | "
          f"LR={lr:.1e} | {elapsed:.1f}s", end="")
    
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'net': model.state_dict(),
            'val_iou': best_val_iou,
        }, f"{SAVE_DIR}/stage1_best.pth.tar")
        print(f" ⭐")
    else:
        print()

print(f"\n✅ Best: epoch {best_epoch}, Val IoU={best_val_iou:.4f}")

✅ Reloaded original weights (pre-deploy). Missing=0, Unexpected=0
✅ RepVGG in training mode (rbr_dense present, rbr_reparam absent)
✅ Trainable: 4.81M (head), Frozen: 2.47M (backbone+fusion)
STAGE 1 TRAINING
Epochs=15, LR=5e-05, Alpha=0.5, L1w=2.0
Batches: train=980, val=105
Ep  1/15 | Train: loss=0.1407 IoU=0.8228 | Val: loss=0.1528 IoU=0.8117 | LR=3.3e-05 | 66.8s ⭐
Ep  2/15 | Train: loss=0.1352 IoU=0.8314 | Val: loss=0.1481 IoU=0.8191 | LR=5.0e-05 | 65.6s ⭐
Ep  3/15 | Train: loss=0.1318 IoU=0.8369 | Val: loss=0.1454 IoU=0.8233 | LR=5.0e-05 | 65.5s ⭐
Ep  4/15 | Train: loss=0.1296 IoU=0.8404 | Val: loss=0.1447 IoU=0.8246 | LR=4.9e-05 | 65.7s ⭐
Ep  5/15 | Train: loss=0.1283 IoU=0.8425 | Val: loss=0.1437 IoU=0.8260 | LR=4.7e-05 | 65.8s ⭐
Ep  6/15 | Train: loss=0.1272 IoU=0.8441 | Val: loss=0.1431 IoU=0.8268 | LR=4.3e-05 | 65.8s ⭐
Ep  7/15 | Train: loss=0.1263 IoU=0.8454 | Val: loss=0.1427 IoU=0.8275 | LR=3.8e-05 | 65.5s ⭐
Ep  8/15 | Train: loss=0.1256 IoU=0.8465 | Val: loss=0.1424 IoU=0.

In [22]:
# OPTIONAL: Quick sanity check 
# Verify trained model outputs are reasonable after deploy

from lib.models.tracker_model import LightFC as LFC
m_check = LFC(cfg, env_num=0, training=False)
s1 = torch.load(f"{WORK}/checkpoints/stage1_best.pth.tar", map_location='cpu')
m_check.load_state_dict(s1['net'], strict=True)
for m in m_check.head.modules():
    if hasattr(m, 'switch_to_deploy'): m.switch_to_deploy()
m_check.eval().cuda()
force_cuda(m_check)

# Test on first sequence init frame
with torch.no_grad():
    z_p, _, z_m = repo_sample_target(frame_rgb, init_bbox, 2.0, output_sz=128)
    s_p, s_rf, s_m = repo_sample_target(frame_rgb, init_bbox, 4.0, output_sz=256)
    z_t = preprocessor.process(z_p, z_m)
    s_t = preprocessor.process(s_p, s_m)
    zf = m_check.forward_backbone(z_t.tensors.cuda())
    out = m_check.forward_tracking(zf, s_t.tensors.cuda())
    
    # Direct pred_boxes
    pb = out['pred_boxes']
    print(f"pred_boxes: {pb.cpu().tolist()}")
    
    # cal_bbox decode
    response = output_window * out['score_map']
    cb = m_check.head.cal_bbox(response, out['size_map'], out['offset_map'])
    cb = cb.view(-1,4).mean(0)
    print(f"cal_bbox:   {cb.cpu().tolist()}")
    print(f"Difference: {(pb[0] - cb).abs().cpu().tolist()}")

del m_check

pred_boxes: [[0.60201495885849, 0.4676384925842285, 0.48765239119529724, 0.17912188172340393]]
cal_bbox:   [0.60201495885849, 0.4676384925842285, 0.48765239119529724, 0.17912188172340393]
Difference: [0.0, 0.0, 0.0, 0.0]


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5: INFERENCE WITH STAGE 1 TRAINED WEIGHTS
# ══════════════════════════════════════════════════════════════════════
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │                    PURPOSE                                          │
# │                                                                     │
# │  Run the TRAINED model (stage1_best.pth.tar) on all 89 test       │
# │  sequences and generate submission.csv for Kaggle.                 │
# │                                                                     │
# │  Pipeline: Load weights → Deploy → Track → Write CSV              │
# │  Score: 0.7292 (+0.027 over baseline)                             │
# └─────────────────────────────────────────────────────────────────────┘
#
# ── THREE-STEP MODEL PREPARATION ───────────────────────────────────
#
#   Step 6a: Build → Load → Deploy
#
#   1. BUILD fresh model:
#      model_inf = LightFC(cfg, training=False)
#      → Creates empty backbone + fusion + head
#      → training=False just sets a flag, doesn't affect architecture
#
#   2. LOAD trained weights:
#      model_inf.load_state_dict(stage1_best['net'])
#      → Weights are in TRAINING mode (rbr_dense + rbr_3x3)
#      → Must deploy before inference for correct behavior
#
#   3. DEPLOY (switch_to_deploy):
#      → Fuses rbr_dense + rbr_3x3 into rbr_reparam
#      → Deletes training-only branches
#      → 2× faster inference, identical outputs
#      → NEVER train after this point
#
#   Verification:
#      has_reparam = True  ← deployed (fused conv)
#      has_dense = False   ← training branches removed
#
# ── TRACKING LOOP ──────────────────────────────────────────────────
#
#   For each of 89 test sequences:
#
#   Frame 0: Initialize
#     Read init_bbox from annotation.txt (first line)
#     sample_target(image, init_bbox, factor=2.0, sz=128) → template
#     Preprocessor → normalize → z_feat = backbone(template)
#     Output: init_bbox (unchanged, frame 0 prediction = init bbox)
#
#   Frame 1..N: Track
#     sample_target(image, prev_state, factor=4.0, sz=256) → search
#     Preprocessor → normalize → forward_tracking(z_feat, search)
#     response = hann2d × score_map
#     cal_bbox → scale → map_box_back → clip_box → new_state
#
#   Key differences from Cell 2 (baseline):
#     SAME pipeline, SAME functions, SAME decode
#     ONLY difference: model weights (trained vs pretrained)
#
# ── CSV GENERATION ─────────────────────────────────────────────────
#
#   Reads sample_submission.csv for exact row IDs and order
#   For each row "dataset1/Car_video_42":
#     Parse: seq="dataset1/Car_video", frame=42
#     Look up all_predictions[seq][42]
#     Write: "dataset1/Car_video_42,x,y,w,h"
#
#   Format: 2 decimal places (f"{x:.2f}")
#   Verified: 2 decimal places has zero measurable impact on score
#
# ── COMPLETE FLOW DIAGRAM ──────────────────────────────────────────
#
#   Cell 1: Setup + Explore + Build model (deployed) + Verify
#     ↓
#   Cell 3: Generate training pairs (template+search+GT+OD)
#     ↓      with moderate jitter, saved as .pt chunks
#     ↓
#   Cell 4: Load chunks into Dataset + DataLoader
#     ↓
#   Cell 5: Train head with distillation loss
#     ↓      (reload original weights → train → save best)
#     ↓
#   Sanity: Verify pred_boxes == cal_bbox
#     ↓
#   Cell 6: Inference with trained weights → 0.7292
#     ↓
#   Submit submission.csv to Kaggle
#
# ══════════════════════════════════════════════════════════════
# CELL 5: Inference with Stage 1 Trained Weights
# ══════════════════════════════════════════════════════════════
#
# Steps:
#   1. Build fresh model
#   2. Load stage1_best weights
#   3. switch_to_deploy (fuse RepVGG branches)
#   4. Track all test sequences using native pipeline
#   5. Write submission CSV
# ══════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════
# CELL 5 v6-conservative: EMA + ANCHOR ON, MULTI-SCALE OFF
# ══════════════════════════════════════════════════════════════

import torch, os, csv, time, cv2, json
import numpy as np

STAGE1_CKPT = "{SAVE_DIR}/stage1_best.pth.tar"

# ── Load + Deploy ──────────────────────────────────────────────
from lib.models.tracker_model import LightFC

model_inf = LightFC(cfg, env_num=0, training=False)
s1_ckpt = torch.load(STAGE1_CKPT, map_location='cpu')
model_inf.load_state_dict(s1_ckpt['net'], strict=True)
print(f"✅ Loaded (epoch {s1_ckpt['epoch']}, val_IoU={s1_ckpt['val_iou']:.4f})")

for m in model_inf.modules():
    if hasattr(m, 'switch_to_deploy'):
        m.switch_to_deploy()
model_inf.eval().cuda()
force_cuda(model_inf)
print(f"✅ Deployed")

# ── Imports ────────────────────────────────────────────────────
from lib.test.tracker.data_utils import Preprocessor
from lib.train.data.processing_utils import sample_target as repo_sample_target
from lib.test.utils.hann import hann2d
from lib.utils.box_ops import clip_box

preprocessor = Preprocessor()
hann_window = hann2d(torch.tensor([16, 16]).long(), centered=True).cuda()

TEMPLATE_SZ = 128
SEARCH_SZ = 256
TEMPLATE_F = 2.0
SEARCH_F = 4.0

# ── cal_bbox ───────────────────────────────────────────────────
def cal_bbox(score_map_ctr, size_map, offset_map, return_score=False):
    max_score, idx = torch.max(score_map_ctr.flatten(1), dim=1, keepdim=True)
    idx_y = idx // score_map_ctr.shape[-1]
    idx_x = idx % score_map_ctr.shape[-1]

    # idx is (B, 1) — expand to (B, 2, 1) for gathering ONE value per channel
    idx_expanded = idx.unsqueeze(1).expand(-1, size_map.size(1), -1)

    s = size_map.flatten(2).gather(2, idx_expanded).squeeze(-1)   # (B, 2)
    o = offset_map.flatten(2).gather(2, idx_expanded).squeeze(-1) # (B, 2)

    cx = (idx_x.float() + o[:, 0:1]) / score_map_ctr.shape[-1]   # (B, 1)
    cy = (idx_y.float() + o[:, 1:2]) / score_map_ctr.shape[-1]   # (B, 1)
    w = s[:, 0:1]
    h = s[:, 1:2]

    bbox = torch.cat([cx, cy, w, h], dim=1)  # (B, 4)
    if return_score:
        return bbox, max_score.squeeze()
    return bbox

# ── IoU helper ────────────────────────────────────────────────
def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    if w1 <= 0 or h1 <= 0 or w2 <= 0 or h2 <= 0:
        return 0.0
    xi1 = max(x1, x2); yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2); yi2 = min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0.0

# ── Preprocessor wrapper ──────────────────────────────────────
def preprocess(crop, mask):
    out = preprocessor.process(crop, mask)
    if hasattr(out, 'tensors'):
        return out.tensors.cuda()
    return out.cuda()

# ── EMA update ─────────────────────────────────────────────────
def ema_update(z_old, z_new, alpha):
    if isinstance(z_old, dict):
        return {k: (1 - alpha) * z_old[k] + alpha * z_new[k] for k in z_old}
    else:
        return (1 - alpha) * z_old + alpha * z_new

# ── Decode helper ──────────────────────────────────────────────
def decode_bbox(pred_bbox_tensor, s_rf, prev_bbox, H, W):
    pred = pred_bbox_tensor.squeeze().cpu().numpy()
    pred_img = pred * SEARCH_SZ / s_rf
    prev_x, prev_y, prev_w, prev_h = prev_bbox
    prev_cx = prev_x + prev_w / 2
    prev_cy = prev_y + prev_h / 2
    half_side = SEARCH_SZ / s_rf / 2
    img_cx = pred_img[0] - half_side + prev_cx
    img_cy = pred_img[1] - half_side + prev_cy
    img_w = pred_img[2]
    img_h = pred_img[3]
    img_x = img_cx - img_w / 2
    img_y = img_cy - img_h / 2
    bbox = clip_box(
        [float(img_x), float(img_y), float(img_w), float(img_h)],
        H, W, margin=2
    )
    if isinstance(bbox, (torch.Tensor, np.ndarray)):
        bbox = [float(bbox[i]) for i in range(4)]
    elif not isinstance(bbox, list):
        bbox = list(bbox)
    return bbox

# ══════════════════════════════════════════════════════════════
# CONFIG  (Conservative)
# ══════════════════════════════════════════════════════════════

USE_EMA = True
EMA_ALPHA = 0.002
EMA_SCORE_THRESH = 0.30
EMA_AREA_RATIO_MIN = 0.25
EMA_AREA_RATIO_MAX = 4.0
EMA_IOU_THRESH = 0.4
EMA_UPDATE_EVERY = 3
ANCHOR_WEIGHT = 0.15
USE_MULTISCALE = False        # ← OFF
SCALES = [1.0]                # ← single scale only
LOST_THRESH = 0.15
LOST_COUNT_EXPAND = 5
LOST_SEARCH_F = 6.0

print(f"✅ Config: EMA={USE_EMA} α={EMA_ALPHA} anchor={ANCHOR_WEIGHT}")
print(f"   MultiScale={USE_MULTISCALE} {SCALES}")
print(f"   LostRecovery: thresh={LOST_THRESH} expand_after={LOST_COUNT_EXPAND}")

# ══════════════════════════════════════════════════════════════
# TRACKING
# ══════════════════════════════════════════════════════════════

with open(os.path.join(DATA_ROOT, "metadata/contestant_manifest.json")) as f:
    manifest = json.load(f)
test_manifest = manifest["public_lb"]

results = {}
total_frames = 0
ema_update_count = 0
ema_skip_score = 0
ema_skip_area = 0
ema_skip_iou = 0
ema_skip_interval = 0
lost_recovery_count = 0
t0 = time.time()

for seq_idx, (key, info) in enumerate(test_manifest.items()):
    video_path = os.path.join(DATA_ROOT, info["video_path"])
    ann_path = os.path.join(DATA_ROOT, info["annotation_path"])
    n_frames = info["n_frames"]

    with open(ann_path) as f:
        first_line = f.readline().strip()
    parts = first_line.replace(',', ' ').split()
    init_bbox = [float(parts[i]) for i in range(4)]

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        continue

    seq_ema_alpha = 0.005 if n_frames < 200 else EMA_ALPHA
    seq_preds = []
    state = {'bbox': list(init_bbox), 'lost_count': 0}
    z = None
    z0 = None
    ema_area = init_bbox[2] * init_bbox[3]
    prev_bbox_for_iou = list(init_bbox)
    frames_since_ema = 0

    for frame_idx in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            seq_preds.append([0, 0, 0, 0])
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        H, W = frame_rgb.shape[:2]

        if frame_idx == 0:
            t_crop, t_rf, t_mask = repo_sample_target(
                frame_rgb, init_bbox, TEMPLATE_F, output_sz=TEMPLATE_SZ)
            t_tensor = preprocess(t_crop, t_mask)
            with torch.no_grad():
                z = model_inf.forward_backbone(t_tensor)
            z0 = z.clone()
            seq_preds.append(init_bbox)

        else:
            current_search_f = SEARCH_F
            if state['lost_count'] > LOST_COUNT_EXPAND:
                current_search_f = LOST_SEARCH_F
                lost_recovery_count += 1

            z_track = (1 - ANCHOR_WEIGHT) * z + ANCHOR_WEIGHT * z0

            best_score = float('-inf')
            best_bbox = None
            best_peak = 0.0
            scales = SCALES if USE_MULTISCALE else [1.0]

            for scale in scales:
                try:
                    s_crop, s_rf, s_mask = repo_sample_target(
                        frame_rgb, state['bbox'],
                        current_search_f * scale,
                        output_sz=SEARCH_SZ)
                except:
                    continue
                s_tensor = preprocess(s_crop, s_mask)
                with torch.no_grad():
                    output = model_inf.forward_tracking(z_track, s_tensor)
                response = hann_window * output['score_map']
                pred_bbox, peak_score = cal_bbox(
                    response, output['size_map'], output['offset_map'],
                    return_score=True)
                pk = peak_score.item()
                if pk > best_score:
                    best_score = pk
                    best_peak = pk
                    best_bbox = decode_bbox(
                        pred_bbox, s_rf, state['bbox'], H, W)

            if best_bbox is None:
                best_bbox = list(state['bbox'])
                best_peak = 0.0

            bbox = best_bbox
            prev_bbox_for_iou = list(state['bbox'])
            state['bbox'] = bbox
            seq_preds.append(bbox)
            frames_since_ema += 1

            if best_peak < LOST_THRESH:
                state['lost_count'] = state.get('lost_count', 0) + 1
            else:
                state['lost_count'] = 0

            if USE_EMA:
                if best_peak <= EMA_SCORE_THRESH:
                    ema_skip_score += 1
                elif frames_since_ema < EMA_UPDATE_EVERY:
                    ema_skip_interval += 1
                else:
                    cur_area = bbox[2] * bbox[3]
                    area_ratio = cur_area / max(ema_area, 1e-6)
                    if not (EMA_AREA_RATIO_MIN < area_ratio < EMA_AREA_RATIO_MAX):
                        ema_skip_area += 1
                    elif compute_iou(prev_bbox_for_iou, bbox) <= EMA_IOU_THRESH:
                        ema_skip_iou += 1
                    else:
                        try:
                            new_t_crop, new_t_rf, new_t_mask = repo_sample_target(
                                frame_rgb, bbox, TEMPLATE_F,
                                output_sz=TEMPLATE_SZ)
                            new_t_tensor = preprocess(new_t_crop, new_t_mask)
                            with torch.no_grad():
                                z_new = model_inf.forward_backbone(new_t_tensor)
                            z = ema_update(z, z_new, seq_ema_alpha)
                            ema_update_count += 1
                            frames_since_ema = 0
                            ema_area = 0.9 * ema_area + 0.1 * cur_area
                        except Exception as e:
                            pass

    cap.release()
    results[key] = seq_preds
    total_frames += len(seq_preds)

    if (seq_idx + 1) % 15 == 0:
        elapsed = time.time() - t0
        fps = total_frames / elapsed
        print(f"  {seq_idx+1}/{len(test_manifest)} | "
              f"{total_frames} frames | {fps:.0f} FPS | "
              f"EMA: {ema_update_count} | Lost: {lost_recovery_count}")

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"✅ Done: {total_frames} frames in {elapsed:.0f}s "
      f"({total_frames/elapsed:.0f} FPS)")
print(f"   EMA updates:       {ema_update_count}")
print(f"   EMA skip (score):  {ema_skip_score}")
print(f"   EMA skip (interval): {ema_skip_interval}")
print(f"   EMA skip (area):   {ema_skip_area}")
print(f"   EMA skip (IoU):    {ema_skip_iou}")
print(f"   Lost recoveries:   {lost_recovery_count}")
print(f"{'='*60}")

# ── Write submission ──────────────────────────────────────────
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, "metadata/sample_submission.csv")
with open(SAMPLE_SUB_PATH) as f:
    reader = csv.reader(f)
    header = next(reader)
    all_rows = [row for row in reader]
sample_ids = [r[0] for r in all_rows]

SUB_PATH = f"{WORK}/submission_v6_conservative.csv"
matched = 0
with open(SUB_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row_id in sample_ids:
        seq, frame_str = row_id.rsplit('_', 1)
        frame_idx = int(frame_str)
        preds = results.get(seq, [])
        if 0 <= frame_idx < len(preds):
            x, y, w, h = preds[frame_idx]
            writer.writerow([row_id, f"{x:.2f}", f"{y:.2f}",
                           f"{w:.2f}", f"{h:.2f}"])
            matched += 1
        elif preds:
            x, y, w, h = preds[-1]
            writer.writerow([row_id, f"{x:.2f}", f"{y:.2f}",
                           f"{w:.2f}", f"{h:.2f}"])
            matched += 1
        else:
            writer.writerow([row_id, "0.00", "0.00", "0.00", "0.00"])

print(f"\n✅ Submission: {SUB_PATH}")
print(f"   Matched: {matched}/{len(sample_ids)}")

import shutil
shutil.copy2(SUB_PATH, "/kaggle/working/submission.csv")
print(f"✅ Copied to /kaggle/working/submission.csv")

# cell 6 output 
#### ✅ Loaded (epoch 14, val_IoU=0.8287)
#### ✅ Deployed
#### ✅ Config: EMA=True α=0.002 anchor=0.15
####   MultiScale=False [1.0]
####   LostRecovery: thresh=0.15 expand_after=5
####  15/89 | 5211 frames | 43 FPS | EMA: 1613 | Lost: 8
####  30/89 | 9235 frames | 45 FPS | EMA: 2849 | Lost: 9
####  45/89 | 24222 frames | 48 FPS | EMA: 7402 | Lost: 13
####  60/89 | 47063 frames | 48 FPS | EMA: 13295 | Lost: 87
####  75/89 | 61817 frames | 48 FPS | EMA: 17659 | Lost: 115

#### ============================================================
#### ✅ Done: 74293 frames in 1534s (48 FPS)
####   EMA updates:       21517
####   EMA skip (score):  5734
####   EMA skip (interval): 42848
####   EMA skip (area):   3490
####   EMA skip (IoU):    615
####   Lost recoveries:   133
#### ============================================================

#### ✅ Submission: /kaggle/working/submission_v6_conservative.csv
####   Matched: 74293/74293